 # 00. EXP11B — Ringkasan Eksperimen



 Bagian ini menjelaskan tujuan EXP11B. Eksperimen ini melanjutkan EXP11A sebagai best terbaru

 dan hanya mengubah risk function MBR decoder. Ordinal PMF, EXP09D tournament gate,

 dan policy M-only dari EXP11A tetap dipreserve; yang diuji adalah penalty ringan untuk

 outcome, goal difference, total goal, dan underprediction bias.

 # 01. Setup, Seed, Path, dan Guardrail



 Bagian ini menyiapkan library, seed, path input/output, dan konfigurasi eksperimen. Guardrail

 dipakai supaya eksperimen tetap legal, tidak memakai GT/external data, tidak memakai

 XGBoost/TabPFN/tqdm, dan output disimpan ke folder EXP11B.

In [ ]:
import os, json, math, random, warnings, copy
from pathlib import Path
from collections import defaultdict, deque
from itertools import product
import time
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except Exception:
    def display(x): print(x)
from sklearn.metrics import accuracy_score, mean_absolute_error

warnings.filterwarnings('ignore')
SEED = 42

def seed_everything(seed: int = 42) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)
pd.set_option('display.max_columns', 220)
pd.set_option('display.width', 180)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() in {'notebook', 'notebooks'}:
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'data' / 'train.csv').exists() and (PROJECT_ROOT.parent / 'data' / 'train.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'exp11b_mbr_decoder_penalty_tuning'
FIG_DIR = OUTPUT_DIR / 'figures'
PRED_DIR = OUTPUT_DIR / 'predictions'
SUB_DIR = OUTPUT_DIR / 'submissions'
SUM_DIR = OUTPUT_DIR / 'summaries'
for d in [OUTPUT_DIR, FIG_DIR, PRED_DIR, SUB_DIR, SUM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SAMPLE_SUB_PATH = DATA_DIR / 'sample submission.csv'
META_PATH = DATA_DIR / 'metadata.txt'

SCREEN_N_TRIALS = 10
REFINE_N_TRIALS = 20
BOOSTERS_TO_RUN = ['lgb', 'cat']
XGB_POLICY = 'skip'
USE_TABPFN = False
USE_RECURSIVE = False
USE_TEACHER_STUDENT = False
USE_PRETRAINED_MODEL = False
USE_EXTERNAL_DATA = False
USE_GT_AUDIT = False
USE_SUBMISSION_AS_FEATURE = False
USE_XGBOOST = False
USE_PROGRESS_BAR = False
MODEL_VERBOSE = False
LGB_EARLY_STOP_VERBOSE = False
RUN_DATA_DRIVEN_GATE = False  # optional EXP09D V4 remains disabled by default to avoid validation overfit
NOTEBOOK_START_TIME = time.time()
SECTION_TIMES = {}

def log_section(title: str):
    print('\n' + '=' * 100, flush=True)
    print(f'[SECTION] {title}', flush=True)
    print('=' * 100, flush=True)

def log_info(msg: str):
    print(f'[INFO] {msg}', flush=True)

def log_check(name: str, passed: bool, detail: str = ''):
    status = 'PASS' if bool(passed) else 'FAIL'
    suffix = f' | {detail}' if detail else ''
    print(f'[CHECK] {name}: {status}{suffix}', flush=True)

def log_warn(msg: str):
    print(f'[WARN] {msg}', flush=True)

def log_result(msg: str):
    print(f'[RESULT] {msg}', flush=True)

def log_saved(path):
    print(f'[SAVED] {path}', flush=True)

@contextmanager
def timed_section(name: str):
    start = time.time()
    log_section(name)
    try:
        yield
    finally:
        elapsed = time.time() - start
        SECTION_TIMES[name] = elapsed
        print(f'[DONE] {name} finished in {elapsed:.2f}s', flush=True)

class ProgressLogger:
    def __init__(self, total, name='process', every=None, min_interval=5.0):
        self.total = int(total)
        self.name = str(name)
        self.every = int(every or max(1, self.total // 10))
        self.min_interval = float(min_interval)
        self.start = time.time()
        self.last_print = self.start
        self.current = 0
        print(f'[PROGRESS START] {self.name} | total={self.total:,}', flush=True)
    def update(self, current=None, step=1):
        if current is None:
            self.current += step
        else:
            self.current = int(current)
        now = time.time()
        if (self.current % self.every == 0) or ((now - self.last_print) >= self.min_interval) or (self.current >= self.total):
            elapsed = now - self.start
            speed = self.current / elapsed if elapsed > 0 else 0
            remaining = self.total - self.current
            eta = remaining / speed if speed > 0 else 0
            pct = 100 * self.current / self.total if self.total else 100
            print(f'[PROGRESS] {self.name} | {self.current:,}/{self.total:,} | {pct:.1f}% | elapsed={elapsed:.1f}s | speed={speed:.1f} rows/s | eta={eta:.1f}s', flush=True)
            self.last_print = now
    def close(self):
        elapsed = time.time() - self.start
        print(f'[PROGRESS DONE] {self.name} | total={self.total:,} | elapsed={elapsed:.1f}s', flush=True)

print('[INFO] output:', OUTPUT_DIR.resolve())
log_info('EXP11B source of truth: EXP05A-LITE-FIX-V2 / EXP09D / EXP11A. Only MBR risk penalty is changed.')
print('[INFO] XGBoost policy:', XGB_POLICY)
log_check('TabPFN disabled', not USE_TABPFN)
log_check('Recursive update disabled', not USE_RECURSIVE)
log_check('Teacher-student disabled', not USE_TEACHER_STUDENT)
log_check('External data disabled', not USE_EXTERNAL_DATA)
log_check('Progress bar package disabled', not USE_PROGRESS_BAR)

RUN_V3_OPTIONAL = False

def iter_progress(iterable, total=None, desc='process'):
    """Small manual progress wrapper."""
    if total is None:
        try:
            total = len(iterable)
        except Exception:
            total = 0
    progress = ProgressLogger(total or 0, name=desc)
    for i, item in enumerate(iterable, start=1):
        yield item
        progress.update(i)
    progress.close()


 # 02. Load Data dan Basic Audit



 Bagian ini membaca train, test, sample submission, dan metadata. Output yang diharapkan adalah shape

 data, range tanggal, serta pengecekan awal agar format input sesuai pipeline EXP05A.

In [ ]:
def validate_input_files(file_paths: dict) -> None:
    missing = [(name, str(path)) for name, path in file_paths.items() if not Path(path).exists()]
    if missing:
        msg = '\n'.join([f'- {n}: {p}' for n, p in missing])
        raise FileNotFoundError('Missing input files:\n' + msg)

validate_input_files({'train': TRAIN_PATH, 'test': TEST_PATH, 'sample_submission': SAMPLE_SUB_PATH, 'metadata': META_PATH})
train_raw = pd.read_csv(TRAIN_PATH, parse_dates=['date'])
test_raw = pd.read_csv(TEST_PATH, parse_dates=['date'])
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)
with open(META_PATH, 'r', encoding='utf-8') as f:
    metadata_text = f.read()
print('[INFO] train:', train_raw.shape, '| test:', test_raw.shape, '| sample:', sample_submission.shape)
display(train_raw.head(2))


 # 03. Core Helper dari EXP05A



 Bagian ini mempertahankan helper utama dari EXP05A, termasuk evaluator AW-MAE, cleaning row-level,

 fungsi score/outcome, dan utilitas validasi. Bagian ini tidak diubah besar-besaran agar behavior

 baseline tetap preserve.

In [ ]:
def select_unique_columns(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    seen, ordered = set(), []
    for c in cols:
        if c not in seen:
            ordered.append(c); seen.add(c)
    missing = [c for c in ordered if c not in df.columns]
    if missing:
        raise KeyError(f'select_unique_columns missing: {missing[:20]}')
    out = df.loc[:, ordered].copy()
    assert len(out.columns) == len(pd.Index(out.columns).unique()), 'Duplicate columns after select_unique_columns'
    return out

def assert_no_duplicate_columns(df: pd.DataFrame, context: str) -> None:
    if len(df.columns) != len(pd.Index(df.columns).unique()):
        dupes = pd.Index(df.columns)[pd.Index(df.columns).duplicated()].tolist()
        raise AssertionError(f'{context} duplicate columns: {dupes}')

def safe_to_string(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = out[c].astype('string').fillna('__MISSING__')
    return out

def safe_numeric(x):
    return pd.to_numeric(x, errors='coerce')

def safe_divide(a, b):
    out = pd.to_numeric(a, errors='coerce') / pd.to_numeric(b, errors='coerce')
    return out.replace([np.inf, -np.inf], np.nan)

def _outcome(a: int, b: int) -> int:
    a, b = int(a), int(b)
    return 0 if a > b else (1 if a == b else 2)

def get_tournament_weight(tournament: str) -> float:
    t = str(tournament).strip().lower()
    if ('fifa world cup' in t) or (t == 'world cup'):
        return 2.00
    if ('afc championship' in t) or ('afc asian cup' in t) or ('asian cup' in t):
        return 1.80
    if 'friendly' in t:
        return 0.96
    return 1.20

EXACT_PENALTY = 0.30
OUTCOME_PENALTY = 0.25
GD_PENALTY = 0.15
WRONG_OUTCOME_MULTIPLIER = 1.50
NONLINEAR_POWER = 1.50

def official_match_loss(y_team_true, y_opp_true, y_team_pred, y_opp_pred) -> float:
    y_team_true, y_opp_true, y_team_pred, y_opp_pred = map(int, [y_team_true, y_opp_true, y_team_pred, y_opp_pred])
    mae = (abs(y_team_true-y_team_pred) + abs(y_opp_true-y_opp_pred)) / 2
    exact = int(y_team_true == y_team_pred and y_opp_true == y_opp_pred)
    outcome_ok = int(_outcome(y_team_true, y_opp_true) == _outcome(y_team_pred, y_opp_pred))
    gd_ok = int((y_team_true-y_opp_true) == (y_team_pred-y_opp_pred))
    penalty = EXACT_PENALTY*(1-exact) + OUTCOME_PENALTY*(1-outcome_ok) + GD_PENALTY*(1-gd_ok)
    multiplier = 1.0 if outcome_ok else WRONG_OUTCOME_MULTIPLIER
    return float(((mae + penalty) * multiplier) ** NONLINEAR_POWER)

def awmae_score(y_team_true, y_opp_true, y_team_pred, y_opp_pred, tournaments) -> float:
    losses = np.array([official_match_loss(a, b, pa, pb) for a,b,pa,pb in zip(y_team_true, y_opp_true, y_team_pred, y_opp_pred)], dtype=float)
    weights = np.array([get_tournament_weight(t) for t in tournaments], dtype=float)
    return float(np.sum(losses * weights) / np.sum(weights))

def clean_row_level(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'altitude_venue' in out.columns:
        out.loc[out['altitude_venue'] == -9999, 'altitude_venue'] = np.nan
    if 'date' in out.columns:
        out['date'] = pd.to_datetime(out['date'], errors='coerce')
    return safe_to_string(out, ['team','opponent','gender','tournament','venue_country','confederation_team','confederation_opp'])

def build_match_level(df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
    df = df.copy()
    required = ['match_id','team','opponent','date','gender','tournament']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f'build_match_level missing: {missing}')
    bad_counts = df['match_id'].value_counts().loc[lambda s: s != 2]
    if len(bad_counts):
        raise AssertionError(f'Match id not two rows. Example: {bad_counts.head().to_dict()}')
    rows = []
    shared = ['date','gender','tournament','venue_country','neutral','altitude_venue','temperature_venue']
    side_map = {
        'is_home': 'is_home',
        'confederation': 'confederation_team',
        'population': 'population_team',
        'gdp_per_capita': 'gdp_per_capita_team',
        'distance_travel': 'distance_travel_team',
    }
    for mid, grp in iter_progress(df.groupby('match_id', sort=False), total=df['match_id'].nunique(), desc='build_match_level'):
        pair = grp.sort_values(['team','opponent']).reset_index(drop=True)
        a, b = pair.iloc[0], pair.iloc[1]
        row = {'match_id': mid, 'team_a': a['team'], 'team_b': b['team']}
        for c in shared:
            if c in pair.columns:
                vals = pair[c].dropna().unique().tolist()
                row[c] = vals[0] if vals else np.nan
        if 'Id' in pair.columns:
            row['row_id_a'], row['row_id_b'] = a['Id'], b['Id']
        for name, col in side_map.items():
            row[f'team_a_{name}'] = a[col] if col in pair.columns else np.nan
            row[f'team_b_{name}'] = b[col] if col in pair.columns else np.nan
        if is_train:
            row['team_a_goals'] = a['team_goals']
            row['team_b_goals'] = b['team_goals']
        rows.append(row)
    out = pd.DataFrame(rows).sort_values(['date','match_id']).reset_index(drop=True)
    assert_no_duplicate_columns(out, 'match_level')
    return out

def make_time_based_holdout(train_match: pd.DataFrame, valid_fraction: float = 0.2):
    df = train_match.sort_values(['date','match_id']).reset_index(drop=True)
    n_valid = max(1, int(len(df)*valid_fraction))
    tr, va = df.iloc[:-n_valid].copy(), df.iloc[-n_valid:].copy()
    assert set(tr['match_id']).isdisjoint(set(va['match_id']))
    print('[INFO] train fold:', tr.shape, tr['date'].min(), '->', tr['date'].max())
    print('[INFO] valid fold:', va.shape, va['date'].min(), '->', va['date'].max())
    return tr, va


 # 04. Canonical Match-Level Builder



 Dataset berisi dua row untuk satu match. Bagian ini mengubah data row-level menjadi match-level agar

 prediksi team_a dan team_b konsisten, lalu nanti bisa dikembalikan ke format submission.

In [ ]:
train_clean = clean_row_level(train_raw)
test_clean = clean_row_level(test_raw)
train_match_base = build_match_level(train_clean, is_train=True)
test_match_base = build_match_level(test_clean, is_train=False)
print('[INFO] train_match_base:', train_match_base.shape)
print('[INFO] test_match_base :', test_match_base.shape)
display(train_match_base.head(3))


 # 05. Static dan History Features dari EXP05A



 Bagian ini membangun fitur utama EXP05A, termasuk static features dan history-full legal features.

 Feature set ini menjadi baseline untuk goal head, tail head, dan baseline outcome head.

In [ ]:
def engineer_static_match_features(df_match: pd.DataFrame) -> pd.DataFrame:
    df = df_match.copy()
    df = safe_to_string(df, ['team_a','team_b','gender','tournament','venue_country','team_a_confederation','team_b_confederation'])
    df['pair_key'] = df['team_a'].astype(str) + '__VS__' + df['team_b'].astype(str)
    df['confed_pair_key'] = df['team_a_confederation'].astype(str) + '__VS__' + df['team_b_confederation'].astype(str)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['match_year'] = df['date'].dt.year
    df['match_month'] = df['date'].dt.month
    df['match_quarter'] = df['date'].dt.quarter
    df['match_dayofweek'] = df['date'].dt.dayofweek
    df['match_dayofyear'] = df['date'].dt.dayofyear
    df['match_is_weekend'] = (df['match_dayofweek'] >= 5).astype(int)
    df['match_decade'] = (df['match_year'] // 10) * 10
    neutral = safe_numeric(df.get('neutral', 0)).fillna(0)
    home_a = safe_numeric(df.get('team_a_is_home', 0)).fillna(0)
    home_b = safe_numeric(df.get('team_b_is_home', 0)).fillna(0)
    df['home_side'] = np.where(neutral == 1, 0, np.where(home_a == 1, 1, np.where(home_b == 1, -1, 0)))
    df['same_confederation'] = (df['team_a_confederation'].astype(str) == df['team_b_confederation'].astype(str)).astype(int)
    t = df['tournament'].astype(str).str.lower()
    df['is_friendly'] = t.str.contains('friendly').astype(int)
    df['is_world_cup'] = (t.str.contains('fifa world cup') | (t == 'world cup')).astype(int)
    df['is_qualification'] = t.str.contains('qual').astype(int)
    df['is_nations_league'] = t.str.contains('nations').astype(int)
    df['tournament_weight_proxy'] = df['tournament'].map(get_tournament_weight)
    for base, ca, cb in [('population','team_a_population','team_b_population'), ('gdp_per_capita','team_a_gdp_per_capita','team_b_gdp_per_capita'), ('distance_travel','team_a_distance_travel','team_b_distance_travel')]:
        df[ca], df[cb] = safe_numeric(df.get(ca)), safe_numeric(df.get(cb))
        df[f'{base}_diff'] = df[ca] - df[cb]
        df[f'{base}_abs_diff'] = (df[ca] - df[cb]).abs()
        df[f'log_{base}_a'] = np.log1p(df[ca].clip(lower=0))
        df[f'log_{base}_b'] = np.log1p(df[cb].clip(lower=0))
        df[f'log_{base}_diff'] = df[f'log_{base}_a'] - df[f'log_{base}_b']
        df[f'{base}_ratio_ab'] = safe_divide(df[ca], df[cb])
    for c in ['altitude_venue','temperature_venue','neutral']:
        if c in df.columns:
            df[c] = safe_numeric(df[c])
    assert_no_duplicate_columns(df, 'static_features')
    return df

def init_team_state():
    return {'matches_played':0, 'last_match_date':pd.NaT, 'elo_overall':1500.0, 'elo_goal_diff':0.0, 'ewm_points':1.0, 'ewm_gf':1.2, 'ewm_ga':1.2, 'ewm_gd':0.0,
            'points_last5':deque(maxlen=5), 'points_last10':deque(maxlen=10), 'gf_last5':deque(maxlen=5), 'ga_last5':deque(maxlen=5), 'gd_last5':deque(maxlen=5),
            'results_last10':deque(maxlen=10), 'clean_sheet_last5':deque(maxlen=5), 'failed_to_score_last5':deque(maxlen=5)}

def init_h2h_state():
    return {'matches_played':0, 'points_a_last3':deque(maxlen=3), 'gd_a_last3':deque(maxlen=3), 'total_goals_last3':deque(maxlen=3)}

def avg_deque(dq, default=np.nan):
    return float(np.mean(list(dq))) if len(dq) else default

def expected_elo_result(rating_a, rating_b, home_bonus=0.0):
    return 1 / (1 + 10 ** (-((rating_a + home_bonus - rating_b) / 400)))

def snapshot_team_features_full(st, match_date, side):
    days = np.nan if pd.isna(st['last_match_date']) or pd.isna(match_date) else (pd.Timestamp(match_date) - pd.Timestamp(st['last_match_date'])).days
    res = list(st['results_last10'])
    return {f'hist_matches_played_{side}':st['matches_played'], f'hist_elo_overall_{side}':st['elo_overall'], f'hist_elo_gd_{side}':st['elo_goal_diff'],
            f'hist_ewm_points_{side}':st['ewm_points'], f'hist_ewm_gf_{side}':st['ewm_gf'], f'hist_ewm_ga_{side}':st['ewm_ga'], f'hist_ewm_gd_{side}':st['ewm_gd'],
            f'hist_points_avg_last5_{side}':avg_deque(st['points_last5']), f'hist_points_avg_last10_{side}':avg_deque(st['points_last10']),
            f'hist_gf_avg_last5_{side}':avg_deque(st['gf_last5']), f'hist_ga_avg_last5_{side}':avg_deque(st['ga_last5']), f'hist_gd_avg_last5_{side}':avg_deque(st['gd_last5']),
            f'hist_win_rate_last10_{side}':float(np.mean([r==1 for r in res])) if res else np.nan,
            f'hist_draw_rate_last10_{side}':float(np.mean([r==0 for r in res])) if res else np.nan,
            f'hist_loss_rate_last10_{side}':float(np.mean([r==-1 for r in res])) if res else np.nan,
            f'hist_clean_sheet_rate_last5_{side}':avg_deque(st['clean_sheet_last5']), f'hist_failed_to_score_rate_last5_{side}':avg_deque(st['failed_to_score_last5']),
            f'hist_days_since_last_match_{side}':days, f'hist_has_history_{side}':int(st['matches_played']>0)}

def snapshot_h2h_features(st):
    return {'h2h_matches_played_pre':st['matches_played'], 'h2h_points_a_avg_last3':avg_deque(st['points_a_last3']), 'h2h_gd_a_avg_last3':avg_deque(st['gd_a_last3']), 'h2h_total_goals_avg_last3':avg_deque(st['total_goals_last3']), 'h2h_has_history':int(st['matches_played']>0)}

def build_history_feature_row(row, team_states, h2h_states):
    gender, ta, tb, date = str(row['gender']), str(row['team_a']), str(row['team_b']), pd.Timestamp(row['date'])
    sa, sb = team_states[(gender, ta)], team_states[(gender, tb)]
    h2h = h2h_states[(gender, ta, tb)]
    out = {'match_id': row['match_id']}
    out.update(snapshot_team_features_full(sa, date, 'a'))
    out.update(snapshot_team_features_full(sb, date, 'b'))
    out.update(snapshot_h2h_features(h2h))
    for base in ['hist_matches_played','hist_elo_overall','hist_elo_gd','hist_ewm_points','hist_ewm_gf','hist_ewm_ga','hist_ewm_gd','hist_points_avg_last5','hist_points_avg_last10','hist_gf_avg_last5','hist_ga_avg_last5','hist_gd_avg_last5','hist_win_rate_last10','hist_draw_rate_last10','hist_loss_rate_last10','hist_clean_sheet_rate_last5','hist_failed_to_score_rate_last5']:
        a, b = out.get(f'{base}_a'), out.get(f'{base}_b')
        out[f'{base}_diff'] = a - b if pd.notna(a) and pd.notna(b) else np.nan
        out[f'{base}_abs_diff'] = abs(a - b) if pd.notna(a) and pd.notna(b) else np.nan
    a, b = out.get('hist_days_since_last_match_a'), out.get('hist_days_since_last_match_b')
    out['hist_rest_days_diff'] = a - b if pd.notna(a) and pd.notna(b) else np.nan
    return out

def update_states_from_score(sa, sb, h2h, ctx, ga, gb):
    date = pd.Timestamp(ctx['date']); weight = get_tournament_weight(ctx.get('tournament','')); home_side = int(ctx.get('home_side',0) or 0)
    exp_a = expected_elo_result(sa['elo_overall'], sb['elo_overall'], 60*home_side)
    act_a = 1 if ga > gb else (0.5 if ga == gb else 0)
    delta = 24 * weight * (act_a - exp_a)
    sa['elo_overall'] += delta; sb['elo_overall'] -= delta
    gd = ga - gb
    resid = gd - ((sa['elo_goal_diff'] - sb['elo_goal_diff'] + 10*home_side)/100)
    sa['elo_goal_diff'] += 6*resid; sb['elo_goal_diff'] -= 6*resid
    pa = 3 if ga > gb else (1 if ga == gb else 0); pb = 3 if gb > ga else (1 if ga == gb else 0)
    ra = 1 if ga > gb else (0 if ga == gb else -1); rb = 1 if gb > ga else (0 if ga == gb else -1)
    for st, pts, gf, gc, res in [(sa,pa,ga,gb,ra),(sb,pb,gb,ga,rb)]:
        st['matches_played'] += 1; st['last_match_date'] = date
        alpha = 0.35
        st['ewm_points'] = alpha*pts + (1-alpha)*st['ewm_points']; st['ewm_gf'] = alpha*gf + (1-alpha)*st['ewm_gf']; st['ewm_ga'] = alpha*gc + (1-alpha)*st['ewm_ga']; st['ewm_gd'] = alpha*(gf-gc) + (1-alpha)*st['ewm_gd']
        st['points_last5'].append(pts); st['points_last10'].append(pts); st['gf_last5'].append(gf); st['ga_last5'].append(gc); st['gd_last5'].append(gf-gc); st['results_last10'].append(res); st['clean_sheet_last5'].append(int(gc==0)); st['failed_to_score_last5'].append(int(gf==0))
    h2h['matches_played'] += 1; h2h['points_a_last3'].append(pa); h2h['gd_a_last3'].append(gd); h2h['total_goals_last3'].append(ga+gb)

def build_train_history_features_full(match_df):
    df = match_df.sort_values(['date','match_id']).reset_index(drop=True)
    team_states, h2h_states, rows = defaultdict(init_team_state), defaultdict(init_h2h_state), []
    for _, row in iter_progress(df.iterrows(), total=len(df), desc='build_train_history_full'):
        rows.append(build_history_feature_row(row, team_states, h2h_states))
        ka, kb = (str(row['gender']),str(row['team_a'])), (str(row['gender']),str(row['team_b']))
        kh = (str(row['gender']),str(row['team_a']),str(row['team_b']))
        update_states_from_score(team_states[ka], team_states[kb], h2h_states[kh], {'date':row['date'], 'tournament':row.get('tournament',''), 'home_side':row.get('home_side',0)}, int(row['team_a_goals']), int(row['team_b_goals']))
    return pd.DataFrame(rows), copy.deepcopy(dict(team_states)), copy.deepcopy(dict(h2h_states))

def build_future_history_features_freeze(future_df, team_states_cutoff, h2h_states_cutoff):
    df = future_df.sort_values(['date','match_id']).reset_index(drop=True)
    team_states, h2h_states = defaultdict(init_team_state), defaultdict(init_h2h_state)
    team_states.update(copy.deepcopy(team_states_cutoff)); h2h_states.update(copy.deepcopy(h2h_states_cutoff))
    rows = []
    for _, row in iter_progress(df.iterrows(), total=len(df), desc='build_future_history_freeze'):
        rows.append(build_history_feature_row(row, team_states, h2h_states))
        for team in [str(row['team_a']), str(row['team_b'])]:
            team_states[(str(row['gender']), team)]['last_match_date'] = pd.Timestamp(row['date'])
    return pd.DataFrame(rows)

train_static_all = engineer_static_match_features(train_match_base)
test_static_features = engineer_static_match_features(test_match_base)
full_train_history_all, cutoff_team_states_full, cutoff_h2h_states_full = build_train_history_features_full(train_static_all)
train_feature_all = train_static_all.merge(full_train_history_all, on='match_id', how='left', validate='one_to_one')
full_test_history = build_future_history_features_freeze(test_static_features, cutoff_team_states_full, cutoff_h2h_states_full)
test_feature_full = test_static_features.merge(full_test_history, on='match_id', how='left', validate='one_to_one')

train_fold_base, valid_fold_base = make_time_based_holdout(train_match_base, valid_fraction=0.2)
train_fold_static = train_static_all[train_static_all['match_id'].isin(train_fold_base['match_id'])].copy()
valid_fold_static = train_static_all[train_static_all['match_id'].isin(valid_fold_base['match_id'])].copy()
train_fold_hist, cutoff_team_states_valid, cutoff_h2h_states_valid = build_train_history_features_full(train_fold_static)
valid_hist = build_future_history_features_freeze(valid_fold_static, cutoff_team_states_valid, cutoff_h2h_states_valid)
train_feature_full = train_fold_static.merge(train_fold_hist, on='match_id', how='left', validate='one_to_one')
valid_feature_full = valid_fold_static.merge(valid_hist, on='match_id', how='left', validate='one_to_one')

assert_no_duplicate_columns(train_feature_full, 'train_feature_full')
assert_no_duplicate_columns(valid_feature_full, 'valid_feature_full')
print('[INFO] train_feature_full:', train_feature_full.shape)
print('[INFO] valid_feature_full:', valid_feature_full.shape)


 # 06. EXP09A Feature Repair dan EXP09D Tournament Gate



 Bagian ini menambahkan fungsi feature repair dari EXP09A seperti GDP imputation, tournament

 structure, cold-start flags, women temporal trend, dan confederation interaction. Pada EXP09D,

 fitur ini hanya memengaruhi outcome repair model.

In [ ]:
student_categorical_features = [
    'team_a', 'team_b', 'gender', 'tournament', 'venue_country',
    'team_a_confederation', 'team_b_confederation', 'pair_key', 'confed_pair_key'
]
student_numeric_features_static = [
    'neutral', 'altitude_venue', 'temperature_venue',
    'team_a_is_home', 'team_b_is_home',
    'team_a_population', 'team_b_population',
    'team_a_gdp_per_capita', 'team_b_gdp_per_capita',
    'team_a_distance_travel', 'team_b_distance_travel',
    'match_year', 'match_month', 'match_quarter', 'match_dayofweek',
    'match_dayofyear', 'match_is_weekend', 'match_decade',
    'home_side', 'same_confederation',
    'is_friendly', 'is_world_cup', 'is_qualification',
    'is_nations_league', 'tournament_weight_proxy',
    'population_diff', 'population_abs_diff',
    'log_population_a', 'log_population_b', 'log_population_diff',
    'population_ratio_ab',
    'gdp_per_capita_diff', 'gdp_per_capita_abs_diff',
    'log_gdp_per_capita_a', 'log_gdp_per_capita_b',
    'log_gdp_per_capita_diff', 'gdp_per_capita_ratio_ab',
    'distance_travel_diff', 'distance_travel_abs_diff',
    'log_distance_travel_a', 'log_distance_travel_b',
    'log_distance_travel_diff', 'distance_travel_ratio_ab',
]

def derive_exp05a_feature_columns(df):
    history_numeric_features = [c for c in df.columns if c.startswith('hist_') or c.startswith('h2h_')]
    feature_cols = [
        c for c in (student_categorical_features + student_numeric_features_static + history_numeric_features)
        if c in df.columns
    ]
    cat_features = [c for c in feature_cols if c in student_categorical_features]
    return feature_cols, cat_features

# EXP05A feature set is captured BEFORE the EXP09A repair patch.
exp05a_feature_cols, exp05a_cat_features = derive_exp05a_feature_columns(train_feature_full)
log_result(f'EXP05A original feature count: {len(exp05a_feature_cols)}')
log_result(f'EXP05A original categorical count: {len(exp05a_cat_features)}')



with timed_section('exp09c_feature_repair_patch'):
    EXP09A_ADDED_FEATURE_COLS = []
    EXP09A_ADDED_CAT_COLS = []
    EXP09A_PATCH_GROUPS = {}

    def _append_unique(base, items):
        out = list(base)
        for x in items:
            if x not in out:
                out.append(x)
        return out

    def _safe_num_col(df, col):
        if col in df.columns:
            return pd.to_numeric(df[col], errors='coerce')
        return pd.Series(np.nan, index=df.index)

    def _latest_team_gdp_lookup(fit_df):
        rows = []
        for side in ['a','b']:
            team_col = f'team_{side}'
            gdp_col = f'team_{side}_gdp_per_capita'
            conf_col = f'team_{side}_confederation'
            if team_col in fit_df.columns and gdp_col in fit_df.columns:
                tmp = fit_df[[team_col, 'gender', 'date', gdp_col] + ([conf_col] if conf_col in fit_df.columns else [])].copy()
                tmp = tmp.rename(columns={team_col:'team', gdp_col:'gdp', conf_col:'confederation'})
                rows.append(tmp)
        if not rows:
            return {}, pd.DataFrame()
        long = pd.concat(rows, ignore_index=True)
        long['date'] = pd.to_datetime(long['date'], errors='coerce')
        long['gdp'] = pd.to_numeric(long['gdp'], errors='coerce')
        long = long.dropna(subset=['gdp']).sort_values('date')
        latest = long.groupby(['gender','team'])['gdp'].last().to_dict()
        return latest, long

    def add_exp09a_gdp_imputation_features(train_df, test_df, feature_cols=None, cat_cols=None):
        fit_df = train_df.copy()
        latest_lookup, long_gdp = _latest_team_gdp_lookup(fit_df)
        global_median = float(long_gdp['gdp'].median()) if len(long_gdp) else 0.0
        if not np.isfinite(global_median):
            global_median = 0.0
        by_gender_conf = {}
        if len(long_gdp) and 'confederation' in long_gdp.columns:
            by_gender_conf = long_gdp.groupby(['gender','confederation'])['gdp'].median().to_dict()
        by_gender = long_gdp.groupby('gender')['gdp'].median().to_dict() if len(long_gdp) else {}

        def patch_one(df):
            out = df.copy()
            new_cols = []
            for side in ['a','b']:
                team_col = f'team_{side}'
                raw_col = f'team_{side}_gdp_per_capita'
                conf_col = f'team_{side}_confederation'
                raw = _safe_num_col(out, raw_col)
                out[f'gdp_team_{side}_raw'] = raw
                out[f'gdp_team_{side}_was_missing'] = raw.isna().astype(int)

                fill_vals = []
                for _, row in out.iterrows():
                    raw_val = row.get(raw_col, np.nan)
                    raw_val = pd.to_numeric(pd.Series([raw_val]), errors='coerce').iloc[0]
                    if pd.notna(raw_val):
                        fill_vals.append(float(raw_val)); continue
                    key = (str(row.get('gender','__MISSING__')), str(row.get(team_col,'__MISSING__')))
                    if key in latest_lookup:
                        fill_vals.append(float(latest_lookup[key])); continue
                    gc_key = (str(row.get('gender','__MISSING__')), str(row.get(conf_col,'__MISSING__')))
                    if gc_key in by_gender_conf and pd.notna(by_gender_conf[gc_key]):
                        fill_vals.append(float(by_gender_conf[gc_key])); continue
                    g_key = str(row.get('gender','__MISSING__'))
                    if g_key in by_gender and pd.notna(by_gender[g_key]):
                        fill_vals.append(float(by_gender[g_key])); continue
                    fill_vals.append(global_median)
                out[f'gdp_team_{side}_imputed'] = pd.Series(fill_vals, index=out.index).astype(float)
                out[f'gdp_team_{side}_was_imputed'] = out[f'gdp_team_{side}_was_missing'].astype(int)
                out[f'log_gdp_team_{side}_imputed'] = np.log1p(out[f'gdp_team_{side}_imputed'].clip(lower=0))
                new_cols += [f'gdp_team_{side}_raw', f'gdp_team_{side}_was_missing', f'gdp_team_{side}_imputed', f'gdp_team_{side}_was_imputed', f'log_gdp_team_{side}_imputed']
            out['both_gdp_missing'] = ((out['gdp_team_a_was_missing'] == 1) & (out['gdp_team_b_was_missing'] == 1)).astype(int)
            out['gdp_diff_imputed'] = out['gdp_team_a_imputed'] - out['gdp_team_b_imputed']
            out['gdp_abs_diff_imputed'] = out['gdp_diff_imputed'].abs()
            out['gdp_ratio_imputed'] = safe_divide(out['gdp_team_a_imputed'], out['gdp_team_b_imputed'])
            out['log_gdp_diff_imputed'] = out['log_gdp_team_a_imputed'] - out['log_gdp_team_b_imputed']
            new_cols += ['both_gdp_missing','gdp_diff_imputed','gdp_abs_diff_imputed','gdp_ratio_imputed','log_gdp_diff_imputed']
            return out, new_cols

        tr, cols = patch_one(train_df)
        te, _ = patch_one(test_df)
        return tr, te, _append_unique(feature_cols or [], cols), list(cat_cols or []), cols, []

    def infer_tournament_structure(tournament: str, gender: str = None) -> str:
        t = str(tournament).lower().strip()
        if t in ['', 'nan', 'none', '__missing__']:
            return 'unknown'
        knockout_words = ['knockout', 'final', 'semi', 'quarter', 'play-off', 'playoff', 'third place']
        group_words = ['group', 'league', 'round robin']
        is_knock = any(w in t for w in knockout_words)
        is_group = any(w in t for w in group_words)
        if 'friendly' in t:
            return 'friendly'
        if 'qualif' in t or 'qualification' in t or 'qualifier' in t:
            return 'qualifier'
        if 'nations league' in t:
            return 'nations_league_knockout' if is_knock else 'nations_league_group'
        if 'world cup' in t or 'fifa world cup' in t:
            return 'world_cup_knockout' if is_knock else ('world_cup_group' if is_group else 'world_cup_group')
        continental_tokens = ['asian cup','african cup','euro','european championship','gold cup','copa america','afc championship','concacaf','caf','uefa','ofc','saff','aff']
        if any(tok in t for tok in continental_tokens):
            return 'continental_knockout' if is_knock else 'continental_group'
        if 'games' in t or 'olympic' in t or 'pan american' in t:
            return 'regional_games'
        if 'cup' in t or 'championship' in t or 'tournament' in t:
            return 'other_competitive'
        return 'other_competitive'

    def _competition_family(struct):
        s = str(struct)
        if s == 'friendly': return 'friendly'
        if 'qualifier' in s: return 'qualifier'
        if 'nations_league' in s: return 'nations_league'
        if 'world_cup' in s: return 'world_cup'
        if 'continental' in s: return 'continental'
        if 'games' in s: return 'regional_games'
        if s == 'unknown': return 'unknown'
        return 'other_competitive'

    def _stage_type(struct):
        s = str(struct)
        if 'knockout' in s: return 'knockout'
        if 'group' in s or 'league' in s: return 'group_like'
        if 'qualifier' in s: return 'qualifier'
        if s == 'friendly': return 'friendly'
        return 'other'

    def add_exp09a_tournament_structure_features(train_df, test_df, feature_cols=None, cat_cols=None):
        def patch_one(df):
            out = df.copy()
            out['tournament_structure'] = [infer_tournament_structure(t,g) for t,g in zip(out['tournament'], out['gender'])]
            out['competition_family'] = out['tournament_structure'].map(_competition_family).astype(str)
            out['stage_type'] = out['tournament_structure'].map(_stage_type).astype(str)
            out['is_friendly_structure'] = (out['competition_family'] == 'friendly').astype(int)
            out['is_qualifier_structure'] = (out['competition_family'] == 'qualifier').astype(int)
            out['is_competitive_structure'] = (out['competition_family'] != 'friendly').astype(int)
            out['is_nations_league_structure'] = (out['competition_family'] == 'nations_league').astype(int)
            out['is_world_cup_structure'] = (out['competition_family'] == 'world_cup').astype(int)
            out['is_continental_structure'] = (out['competition_family'] == 'continental').astype(int)
            out['is_knockout_like'] = (out['stage_type'] == 'knockout').astype(int)
            out['is_group_like'] = (out['stage_type'] == 'group_like').astype(int)
            for c in ['tournament_structure','competition_family','stage_type']:
                out[c] = out[c].astype('string').fillna('__MISSING__')
            return out
        cat_new = ['tournament_structure','competition_family','stage_type']
        num_new = ['is_friendly_structure','is_qualifier_structure','is_competitive_structure','is_nations_league_structure','is_world_cup_structure','is_continental_structure','is_knockout_like','is_group_like']
        return patch_one(train_df), patch_one(test_df), _append_unique(feature_cols or [], cat_new + num_new), _append_unique(cat_cols or [], cat_new), cat_new + num_new, cat_new

    def add_exp09a_confederation_interaction_features(train_df, test_df, feature_cols=None, cat_cols=None):
        req = ['team_a_confederation','team_b_confederation']
        if not all(c in train_df.columns for c in req):
            log_warn('Confederation columns not found. Skip confederation interaction features.')
            return train_df, test_df, list(feature_cols or []), list(cat_cols or []), [], []
        def patch_one(df):
            out = df.copy()
            a = out['team_a_confederation'].astype('string').fillna('__MISSING__')
            b = out['team_b_confederation'].astype('string').fillna('__MISSING__')
            out['team_a_confederation_exp09a'] = a
            out['team_b_confederation_exp09a'] = b
            out['confed_pair_exp09a'] = a + '__VS__' + b
            out['same_confederation_exp09a'] = (a == b).astype(int)
            if 'tournament_structure' in out.columns:
                out['tournament_structure_x_confed_pair'] = out['tournament_structure'].astype(str) + '__' + out['confed_pair_exp09a'].astype(str)
            else:
                out['tournament_structure_x_confed_pair'] = '__MISSING__'
            if 'is_competitive_structure' in out.columns:
                out['is_competitive_x_same_confed'] = out['is_competitive_structure'].astype(int) * out['same_confederation_exp09a'].astype(int)
            else:
                out['is_competitive_x_same_confed'] = 0
            for c in ['team_a_confederation_exp09a','team_b_confederation_exp09a','confed_pair_exp09a','tournament_structure_x_confed_pair']:
                out[c] = out[c].astype('string').fillna('__MISSING__')
            return out
        cat_new = ['team_a_confederation_exp09a','team_b_confederation_exp09a','confed_pair_exp09a','tournament_structure_x_confed_pair']
        num_new = ['same_confederation_exp09a','is_competitive_x_same_confed']
        return patch_one(train_df), patch_one(test_df), _append_unique(feature_cols or [], cat_new + num_new), _append_unique(cat_cols or [], cat_new), cat_new + num_new, cat_new

    def add_exp09a_cold_start_features(train_df, test_df, feature_cols=None, cat_cols=None, low_threshold=10):
        def patch_one(df):
            out = df.copy()
            a_cnt = _safe_num_col(out, 'hist_matches_played_a').fillna(0)
            b_cnt = _safe_num_col(out, 'hist_matches_played_b').fillna(0)
            out['team_a_history_count_exp09a'] = a_cnt
            out['team_b_history_count_exp09a'] = b_cnt
            out['team_a_is_new_exp09a'] = (a_cnt <= 0).astype(int)
            out['team_b_is_new_exp09a'] = (b_cnt <= 0).astype(int)
            out['team_a_low_history_exp09a'] = (a_cnt < low_threshold).astype(int)
            out['team_b_low_history_exp09a'] = (b_cnt < low_threshold).astype(int)
            out['any_team_new_exp09a'] = ((out['team_a_is_new_exp09a'] == 1) | (out['team_b_is_new_exp09a'] == 1)).astype(int)
            out['both_team_new_exp09a'] = ((out['team_a_is_new_exp09a'] == 1) & (out['team_b_is_new_exp09a'] == 1)).astype(int)
            out['any_team_low_history_exp09a'] = ((out['team_a_low_history_exp09a'] == 1) | (out['team_b_low_history_exp09a'] == 1)).astype(int)
            out['history_count_min_exp09a'] = np.minimum(a_cnt, b_cnt)
            out['history_count_diff_exp09a'] = a_cnt - b_cnt
            is_w = out['gender'].astype(str).eq('W').astype(int)
            out['team_a_w_history_count_exp09a'] = a_cnt * is_w
            out['team_b_w_history_count_exp09a'] = b_cnt * is_w
            out['team_a_w_low_history_exp09a'] = out['team_a_low_history_exp09a'] * is_w
            out['team_b_w_low_history_exp09a'] = out['team_b_low_history_exp09a'] * is_w
            return out
        cols = ['team_a_history_count_exp09a','team_b_history_count_exp09a','team_a_is_new_exp09a','team_b_is_new_exp09a','team_a_low_history_exp09a','team_b_low_history_exp09a','any_team_new_exp09a','both_team_new_exp09a','any_team_low_history_exp09a','history_count_min_exp09a','history_count_diff_exp09a','team_a_w_history_count_exp09a','team_b_w_history_count_exp09a','team_a_w_low_history_exp09a','team_b_w_low_history_exp09a']
        return patch_one(train_df), patch_one(test_df), _append_unique(feature_cols or [], cols), list(cat_cols or []), cols, []

    def add_exp09a_women_temporal_features(train_df, test_df, feature_cols=None, cat_cols=None):
        def patch_one(df):
            out = df.copy()
            year = pd.to_datetime(out['date'], errors='coerce').dt.year.fillna(out.get('match_year', 0)).astype(float)
            out['year_exp09a'] = year
            out['is_women_exp09a'] = out['gender'].astype(str).eq('W').astype(int)
            out['year_centered_2018_exp09a'] = year - 2018
            out['w_year_centered_exp09a'] = out['is_women_exp09a'] * (year - 2018)
            out['is_post2018_exp09a'] = (year >= 2018).astype(int)
            out['is_post2018_women_exp09a'] = out['is_post2018_exp09a'] * out['is_women_exp09a']
            out['is_post2020_women_exp09a'] = (year >= 2020).astype(int) * out['is_women_exp09a']
            out['era_pre_1990_exp09a'] = (year < 1990).astype(int)
            out['era_1990_2000_exp09a'] = ((year >= 1990) & (year < 2000)).astype(int)
            out['era_2000_2010_exp09a'] = ((year >= 2000) & (year < 2010)).astype(int)
            out['era_2010_2018_exp09a'] = ((year >= 2010) & (year < 2018)).astype(int)
            out['era_post2018_exp09a'] = (year >= 2018).astype(int)
            return out
        cols = ['year_exp09a','is_women_exp09a','year_centered_2018_exp09a','w_year_centered_exp09a','is_post2018_exp09a','is_post2018_women_exp09a','is_post2020_women_exp09a','era_pre_1990_exp09a','era_1990_2000_exp09a','era_2000_2010_exp09a','era_2010_2018_exp09a','era_post2018_exp09a']
        return patch_one(train_df), patch_one(test_df), _append_unique(feature_cols or [], cols), list(cat_cols or []), cols, []

    def run_exp09a_feature_patch(train_df, test_df, feature_cols=None, cat_cols=None):
        feature_cols = list(feature_cols or [])
        cat_cols = list(cat_cols or [])
        added_cols_all, added_cat_all = [], []
        patch_steps = [
            ('gdp_imputation', add_exp09a_gdp_imputation_features),
            ('tournament_structure', add_exp09a_tournament_structure_features),
            ('confederation_interaction', add_exp09a_confederation_interaction_features),
            ('cold_start', add_exp09a_cold_start_features),
            ('women_temporal_trend', add_exp09a_women_temporal_features),
        ]
        tr, te = train_df.copy(), test_df.copy()
        patch_list = []
        for group_name, fn in patch_steps:
            tr, te, feature_cols, cat_cols, added, added_cat = fn(tr, te, feature_cols, cat_cols)
            added_cols_all = _append_unique(added_cols_all, added)
            added_cat_all = _append_unique(added_cat_all, added_cat)
            patch_list.append({'group': group_name, 'n_added_features': len(added), 'added_features': added, 'added_categorical_features': added_cat})
            log_info(f'Added {group_name} features: {len(added)} columns')
        return tr, te, feature_cols, cat_cols, added_cols_all, added_cat_all, patch_list

    # Patch fold train/valid using fold train as fitting source.
    train_feature_full, valid_feature_full, _, _, added_fold, added_cat_fold, patch_list_fold = run_exp09a_feature_patch(train_feature_full, valid_feature_full)

    # Patch full train/test using full train as fitting source.
    train_feature_all, test_feature_full, _, _, added_full, added_cat_full, patch_list_full = run_exp09a_feature_patch(train_feature_all, test_feature_full)

    EXP09A_ADDED_FEATURE_COLS = _append_unique(added_fold, added_full)
    EXP09A_ADDED_CAT_COLS = _append_unique(added_cat_fold, added_cat_full)
    EXP09A_ADDED_NUMERIC_COLS = [c for c in EXP09A_ADDED_FEATURE_COLS if c not in EXP09A_ADDED_CAT_COLS]
    EXP09A_PATCH_GROUPS = patch_list_full

    # Fill categorical missing safely.
    for df_name in ['train_feature_full','valid_feature_full','train_feature_all','test_feature_full']:
        df = globals()[df_name]
        for c in EXP09A_ADDED_CAT_COLS:
            if c in df.columns:
                df[c] = df[c].astype('string').fillna('__MISSING__')
        globals()[df_name] = df

    # GDP imputation audit after patch.
    gdp_imputation_audit_rows = []
    for dataset_name, df in [('train_fold', train_feature_full), ('valid_fold', valid_feature_full), ('full_train', train_feature_all), ('test', test_feature_full)]:
        row = {'dataset': dataset_name, 'n_rows': len(df)}
        for c in ['gdp_team_a_was_imputed','gdp_team_b_was_imputed','both_gdp_missing']:
            row[f'{c}_rate'] = float(pd.to_numeric(df.get(c, 0), errors='coerce').fillna(0).mean())
        gdp_imputation_audit_rows.append(row)
    gdp_imputation_audit = pd.DataFrame(gdp_imputation_audit_rows)
    gdp_imputation_audit.to_csv(SUM_DIR / 'gdp_imputation_audit.csv', index=False)

    patch_summary = []
    for item in EXP09A_PATCH_GROUPS:
        patch_summary.append({
            'feature_group': item['group'],
            'n_added_features': item['n_added_features'],
            'added_features': ', '.join(item['added_features']),
            'added_categorical_features': ', '.join(item['added_categorical_features']),
        })
    feature_patch_summary = pd.DataFrame(patch_summary)
    feature_patch_summary.to_csv(SUM_DIR / 'feature_patch_summary.csv', index=False)
    with open(SUM_DIR / 'exp09a_feature_patch_list.json', 'w', encoding='utf-8') as f:
        json.dump(EXP09A_PATCH_GROUPS, f, indent=2, ensure_ascii=False)

    log_result(f'EXP09A added feature count: {len(EXP09A_ADDED_FEATURE_COLS)}')
    log_result(f'EXP09A added categorical count: {len(EXP09A_ADDED_CAT_COLS)}')
    log_check('Train/valid patch alignment', set(EXP09A_ADDED_FEATURE_COLS).issubset(train_feature_full.columns) and set(EXP09A_ADDED_FEATURE_COLS).issubset(valid_feature_full.columns))
    log_check('Train/test patch alignment', set(EXP09A_ADDED_FEATURE_COLS).issubset(train_feature_all.columns) and set(EXP09A_ADDED_FEATURE_COLS).issubset(test_feature_full.columns))
    log_saved(SUM_DIR / 'exp09a_feature_patch_list.json')
    log_saved(SUM_DIR / 'feature_patch_summary.csv')
    log_saved(SUM_DIR / 'gdp_imputation_audit.csv')
    display(feature_patch_summary)


 # 07. Feature Set Split



 Bagian ini memisahkan EXP05A feature set dan EXP05A+EXP09A feature set. Pemisahan ini penting karena

 goal/tail tetap memakai fitur EXP05A, sedangkan repair outcome memakai fitur tambahan.

In [ ]:
# Build EXP09A feature set AFTER patch. Only the outcome head uses this set in V2.
EXP09A_ADDED_FEATURE_COLS = [c for c in EXP09A_ADDED_FEATURE_COLS if c in train_feature_full.columns]
EXP09A_ADDED_CAT_COLS = [c for c in EXP09A_ADDED_CAT_COLS if c in EXP09A_ADDED_FEATURE_COLS]

exp09a_feature_cols = list(exp05a_feature_cols)
for c in EXP09A_ADDED_FEATURE_COLS:
    if c not in exp09a_feature_cols:
        exp09a_feature_cols.append(c)

exp09a_cat_features = list(exp05a_cat_features)
for c in EXP09A_ADDED_CAT_COLS:
    if c not in exp09a_cat_features:
        exp09a_cat_features.append(c)

# Keep the original EXP05A global names for screening/refine source-of-truth behavior.
student_feature_cols = list(exp05a_feature_cols)
student_cat_features = list(exp05a_cat_features)

feature_columns_before_after = {
    'experiment': 'EXP09C',
    'source_of_truth': 'EXP05A-LITE-FIX-V2',
    'old_feature_cols': exp05a_feature_cols,
    'old_cat_cols': exp05a_cat_features,
    'new_feature_cols': exp09a_feature_cols,
    'new_cat_cols': exp09a_cat_features,
    'added_feature_cols': EXP09A_ADDED_FEATURE_COLS,
    'added_cat_cols': EXP09A_ADDED_CAT_COLS,
}
with open(SUM_DIR / 'feature_columns_before_after.json', 'w', encoding='utf-8') as f:
    json.dump(feature_columns_before_after, f, indent=2, ensure_ascii=False)

with open(SUM_DIR / 'feature_columns.json', 'w', encoding='utf-8') as f:
    json.dump({
        'student_feature_cols_exp05a': exp05a_feature_cols,
        'student_cat_features_exp05a': exp05a_cat_features,
        'student_feature_cols_exp09a': exp09a_feature_cols,
        'student_cat_features_exp09a': exp09a_cat_features,
        'xgb_policy': XGB_POLICY,
    }, f, indent=2, ensure_ascii=False)

with open(SUM_DIR / 'objective_configurations.json', 'w', encoding='utf-8') as f:
    json.dump({'goal':'Poisson', 'outcome':'MultiClass', 'tail':'Binary Logloss', 'xgb':'skipped'}, f, indent=2)
with open(SUM_DIR / 'gender_split_config.json', 'w', encoding='utf-8') as f:
    json.dump({'gender_split': True, 'genders': sorted(train_match_base['gender'].astype(str).unique())}, f, indent=2)

feature_config_df = pd.DataFrame([
    {'head': 'goal_a', 'variant': 'V2_outcome_only', 'feature_set': 'EXP05A', 'n_features': len(exp05a_feature_cols), 'n_cat_features': len(exp05a_cat_features)},
    {'head': 'goal_b', 'variant': 'V2_outcome_only', 'feature_set': 'EXP05A', 'n_features': len(exp05a_feature_cols), 'n_cat_features': len(exp05a_cat_features)},
    {'head': 'outcome', 'variant': 'V2_outcome_only', 'feature_set': 'EXP05A_PLUS_EXP09A', 'n_features': len(exp09a_feature_cols), 'n_cat_features': len(exp09a_cat_features)},
    {'head': 'tail', 'variant': 'V2_outcome_only', 'feature_set': 'EXP05A', 'n_features': len(exp05a_feature_cols), 'n_cat_features': len(exp05a_cat_features)},
])
feature_config_df.to_csv(SUM_DIR / 'head_feature_config.csv', index=False)

exp09c_feature_sets = {
    'V0_baseline': {'goal': 'EXP05A', 'outcome': 'EXP05A', 'tail': 'EXP05A'},
    'V1_allhead': {'goal': 'EXP05A_PLUS_EXP09A', 'outcome': 'EXP05A_PLUS_EXP09A', 'tail': 'EXP05A_PLUS_EXP09A'},
    'V2_outcome_only': {'goal': 'EXP05A', 'outcome': 'EXP05A_PLUS_EXP09A', 'tail': 'EXP05A'},
    'V3_outcome_tail_optional': {'goal': 'EXP05A', 'outcome': 'EXP05A_PLUS_EXP09A', 'tail': 'EXP05A_PLUS_EXP09A'},
}
with open(SUM_DIR / 'exp09c_feature_sets.json', 'w', encoding='utf-8') as f:
    json.dump(exp09c_feature_sets, f, indent=2, ensure_ascii=False)
# Compatibility copy of patch list under EXP09C name.
if 'EXP09A_PATCH_GROUPS' in globals():
    with open(SUM_DIR / 'exp09c_feature_patch_list.json', 'w', encoding='utf-8') as f:
        json.dump(EXP09A_PATCH_GROUPS, f, indent=2, ensure_ascii=False)

log_result(f'EXP09A repaired feature count: {len(exp09a_feature_cols)}')
log_result(f'Added repair feature count: {len(EXP09A_ADDED_FEATURE_COLS)}')
log_check('EXP05A feature set preserved for goal heads', set(exp05a_feature_cols).issubset(set(exp09a_feature_cols)))
log_check('Added categorical features registered', set(EXP09A_ADDED_CAT_COLS).issubset(set(exp09a_cat_features)))
log_saved(SUM_DIR / 'feature_columns_before_after.json')
log_saved(SUM_DIR / 'head_feature_config.csv')
log_saved(SUM_DIR / 'exp09c_feature_sets.json')
display(feature_config_df)


 # 08. EDA Tipis, Feature Patch Summary, dan Target Builder



 Bagian ini menyimpan audit fitur repair serta membangun target supervised seperti goal_a, goal_b,

 outcome, dan tail labels. Outputnya berupa summary feature sebelum/sesudah patch dan dataframe

 target untuk training/validation.

In [ ]:
with timed_section('exp09c_domain_shift_eda'):
    def _year_col(df):
        return pd.to_datetime(df['date'], errors='coerce').dt.year

    def build_gdp_missing_audit(train_df, test_df):
        rows = []
        for name, df in [('train', train_df), ('test', test_df)]:
            tmp = df.copy()
            tmp['year'] = _year_col(tmp)
            g_team = 'gdp_per_capita_team' if 'gdp_per_capita_team' in tmp.columns else None
            g_opp = 'gdp_per_capita_opp' if 'gdp_per_capita_opp' in tmp.columns else None
            for (year, gender), g in tmp.groupby(['year','gender'], dropna=False):
                if g_team and g_opp:
                    mt = pd.to_numeric(g[g_team], errors='coerce').isna()
                    mo = pd.to_numeric(g[g_opp], errors='coerce').isna()
                    rows.append({
                        'dataset': name,
                        'year': year,
                        'gender': gender,
                        'n_rows': len(g),
                        'gdp_team_missing_rate': float(mt.mean()),
                        'gdp_opp_missing_rate': float(mo.mean()),
                        'both_gdp_missing_rate': float((mt & mo).mean()),
                    })
        return pd.DataFrame(rows).sort_values(['dataset','year','gender'])

    def infer_tournament_structure(tournament: str, gender: str = None) -> str:
        t = str(tournament).lower().strip()
        if t in ['', 'nan', 'none', '__missing__']:
            return 'unknown'
        knockout_words = ['knockout', 'final', 'semi', 'quarter', 'play-off', 'playoff', 'third place']
        group_words = ['group', 'league', 'round robin']
        is_knock = any(w in t for w in knockout_words)
        is_group = any(w in t for w in group_words)
        if 'friendly' in t:
            return 'friendly'
        if 'qualif' in t or 'qualification' in t or 'qualifier' in t:
            return 'qualifier'
        if 'nations league' in t:
            return 'nations_league_knockout' if is_knock else 'nations_league_group'
        if 'world cup' in t or 'fifa world cup' in t:
            return 'world_cup_knockout' if is_knock else ('world_cup_group' if is_group else 'world_cup_group')
        continental_tokens = ['asian cup','african cup','euro','european championship','gold cup','copa america','afc championship','concacaf','caf','uefa','ofc','saff','aff']
        if any(tok in t for tok in continental_tokens):
            return 'continental_knockout' if is_knock else 'continental_group'
        if 'games' in t or 'olympic' in t or 'pan american' in t:
            return 'regional_games'
        if 'cup' in t or 'championship' in t or 'tournament' in t:
            return 'other_competitive'
        return 'other_competitive'

    def build_tournament_structure_audit(train_df, test_df):
        rows=[]
        for name, df in [('train', train_df), ('test', test_df)]:
            tmp = df.copy()
            tmp['tournament_structure'] = [infer_tournament_structure(t,g) for t,g in zip(tmp['tournament'], tmp['gender'])]
            cnt = tmp.groupby(['gender','tournament_structure'], dropna=False).size().reset_index(name='n_rows')
            cnt['dataset'] = name
            cnt['share'] = cnt.groupby(['dataset','gender'])['n_rows'].transform(lambda s: s/s.sum())
            rows.append(cnt[['dataset','gender','tournament_structure','n_rows','share']])
        return pd.concat(rows, ignore_index=True)

    def build_cold_start_audit(train_df, test_df, low_threshold=10):
        rows=[]
        for gender in sorted(pd.concat([train_df['gender'], test_df['gender']]).astype(str).dropna().unique()):
            tr_g = train_df[train_df['gender'].astype(str)==gender]
            te_g = test_df[test_df['gender'].astype(str)==gender]
            train_teams = set(tr_g['team'].astype(str))
            test_teams = set(te_g['team'].astype(str))
            hist_count = tr_g.groupby('team').size().to_dict()
            is_new = ~te_g['team'].astype(str).isin(train_teams)
            low_hist = te_g['team'].astype(str).map(hist_count).fillna(0) < low_threshold
            rows.append({
                'gender': gender,
                'n_train_unique_teams': len(train_teams),
                'n_test_unique_teams': len(test_teams),
                'n_test_new_teams': len(test_teams - train_teams),
                'share_test_rows_with_new_team': float(is_new.mean()) if len(te_g) else np.nan,
                'share_test_rows_with_low_history_team': float(low_hist.mean()) if len(te_g) else np.nan,
            })
        return pd.DataFrame(rows)

    def build_women_shift_audit(train_df, test_df):
        rows=[]
        for name, df in [('train', train_df), ('test', test_df)]:
            tmp = df.copy()
            tmp['year'] = _year_col(tmp)
            audit = tmp.groupby(['year','gender'], dropna=False).agg(
                n_rows=('Id','size') if 'Id' in tmp.columns else ('team','size'),
                n_unique_teams=('team','nunique')
            ).reset_index()
            audit['dataset'] = name
            rows.append(audit[['dataset','year','gender','n_rows','n_unique_teams']])
        return pd.concat(rows, ignore_index=True).sort_values(['dataset','year','gender'])

    gdp_missing_audit = build_gdp_missing_audit(train_clean, test_clean)
    tournament_structure_audit = build_tournament_structure_audit(train_clean, test_clean)
    cold_start_audit = build_cold_start_audit(train_clean, test_clean)
    women_shift_audit = build_women_shift_audit(train_clean, test_clean)

    gdp_missing_audit.to_csv(SUM_DIR / 'gdp_missing_audit.csv', index=False)
    tournament_structure_audit.to_csv(SUM_DIR / 'tournament_structure_audit.csv', index=False)
    cold_start_audit.to_csv(SUM_DIR / 'cold_start_audit.csv', index=False)
    women_shift_audit.to_csv(SUM_DIR / 'women_shift_audit.csv', index=False)

    # Lightweight figures.
    try:
        if len(gdp_missing_audit):
            fig, ax = plt.subplots(figsize=(10, 4))
            plot_df = gdp_missing_audit[gdp_missing_audit['dataset'].eq('test')].copy()
            sns.lineplot(data=plot_df, x='year', y='both_gdp_missing_rate', hue='gender', marker='o', ax=ax)
            ax.set_title('Test GDP missing rate by year and gender')
            ax.set_ylabel('Both GDP missing rate')
            fig.tight_layout(); fig.savefig(FIG_DIR / 'gdp_missing_by_year_gender.png', dpi=140); plt.close(fig)
        if len(tournament_structure_audit):
            fig, ax = plt.subplots(figsize=(10, 5))
            plot_df = tournament_structure_audit.groupby(['dataset','tournament_structure'])['n_rows'].sum().reset_index()
            sns.barplot(data=plot_df, x='tournament_structure', y='n_rows', hue='dataset', ax=ax)
            ax.tick_params(axis='x', rotation=45)
            ax.set_title('Tournament structure distribution')
            fig.tight_layout(); fig.savefig(FIG_DIR / 'tournament_structure_distribution.png', dpi=140); plt.close(fig)
        if len(cold_start_audit):
            fig, ax = plt.subplots(figsize=(7, 4))
            sns.barplot(data=cold_start_audit, x='gender', y='share_test_rows_with_new_team', ax=ax)
            ax.set_title('Share of test rows with new team by gender')
            fig.tight_layout(); fig.savefig(FIG_DIR / 'cold_start_team_count_by_gender.png', dpi=140); plt.close(fig)
        if len(women_shift_audit):
            fig, ax = plt.subplots(figsize=(10, 4))
            plot_df = women_shift_audit[women_shift_audit['gender'].astype(str).eq('W')]
            sns.lineplot(data=plot_df, x='year', y='n_rows', hue='dataset', marker='o', ax=ax)
            ax.set_title('Women match row count by year')
            fig.tight_layout(); fig.savefig(FIG_DIR / 'women_match_count_by_year.png', dpi=140); plt.close(fig)
    except Exception as e:
        log_warn(f'EDA plot creation skipped due to: {repr(e)}')

    log_saved(SUM_DIR / 'gdp_missing_audit.csv')
    log_saved(SUM_DIR / 'tournament_structure_audit.csv')
    log_saved(SUM_DIR / 'cold_start_audit.csv')
    log_saved(SUM_DIR / 'women_shift_audit.csv')
    display(gdp_missing_audit.tail(10))
    display(tournament_structure_audit.head(10))
    display(cold_start_audit)


def build_supervised_target_frame(match_df):
    out = pd.DataFrame({
        'match_id': match_df['match_id'].values,
        'y_goal_a': pd.to_numeric(match_df['team_a_goals'], errors='coerce').astype(int).values,
        'y_goal_b': pd.to_numeric(match_df['team_b_goals'], errors='coerce').astype(int).values,
    })
    out['y_outcome'] = [_outcome(a, b) for a, b in zip(out['y_goal_a'], out['y_goal_b'])]
    gd = out['y_goal_a'] - out['y_goal_b']
    total = out['y_goal_a'] + out['y_goal_b']
    out['is_team_a_blowout_5plus'] = (gd >= 5).astype(int)
    out['is_team_b_blowout_5plus'] = (gd <= -5).astype(int)
    out['is_team_a_blowout_7plus'] = (gd >= 7).astype(int)
    out['is_team_b_blowout_7plus'] = (gd <= -7).astype(int)
    out['is_high_total_6plus'] = (total >= 6).astype(int)
    return out

y_train_fold = build_supervised_target_frame(train_fold_base)
y_valid_fold = build_supervised_target_frame(valid_fold_base)
y_full_train = build_supervised_target_frame(train_match_base)

CUTOFF_REGIMES = {'full_history': None, 'modern_cutoff_1990': '1990-01-01'}
with open(SUM_DIR / 'cutoff_regime_config.json', 'w', encoding='utf-8') as f:
    json.dump(CUTOFF_REGIMES, f, indent=2)
with open(SUM_DIR / 'tail_target_definitions.json', 'w', encoding='utf-8') as f:
    json.dump({
        'directional_tail': [
            'is_team_a_blowout_5plus',
            'is_team_b_blowout_5plus',
            'is_team_a_blowout_7plus',
            'is_team_b_blowout_7plus',
            'is_high_total_6plus',
        ]
    }, f, indent=2)

# Keep the original EXP05A global names for screening/refine source-of-truth behavior.
student_feature_cols = list(exp05a_feature_cols)
student_cat_features = list(exp05a_cat_features)

log_info(f'Target frame train fold: {y_train_fold.shape}')
log_info(f'Target frame valid fold: {y_valid_fold.shape}')


 # 09. Model Training Helpers dan Anchor Decode Fix



 Bagian ini berisi helper training CatBoost/LightGBM, fungsi prediksi raw, decoder EXP05A/EXP09C,

 serta fix langsung untuk error lama KeyError gender. valid_decode_df dibuat dari valid_feature_full

 dan hanya merge actual goals.

In [ ]:
try:
    import lightgbm as lgb
except Exception as e:
    lgb = None; print('[WARN] lightgbm import failed:', repr(e))
try:
    from catboost import CatBoostRegressor, CatBoostClassifier
except Exception as e:
    CatBoostRegressor = CatBoostClassifier = None; print('[WARN] catboost import failed:', repr(e))
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except Exception as e:
    optuna = None; print('[WARN] optuna import failed:', repr(e))

def make_default_params(booster, task, seed=42, n_estimators=700):
    if booster == 'cat':
        if task == 'goal': return dict(loss_function='Poisson', eval_metric='Poisson', iterations=n_estimators, learning_rate=0.03, depth=7, l2_leaf_reg=5.0, random_seed=seed, allow_writing_files=False, od_type='Iter', od_wait=100, verbose=MODEL_VERBOSE)
        if task == 'outcome': return dict(loss_function='MultiClass', eval_metric='MultiClass', iterations=n_estimators, learning_rate=0.03, depth=7, l2_leaf_reg=5.0, random_seed=seed, allow_writing_files=False, od_type='Iter', od_wait=100, verbose=MODEL_VERBOSE)
        if task == 'tail': return dict(loss_function='Logloss', eval_metric='Logloss', iterations=n_estimators, learning_rate=0.03, depth=7, l2_leaf_reg=5.0, random_seed=seed, allow_writing_files=False, od_type='Iter', od_wait=100, verbose=MODEL_VERBOSE)
    if booster == 'lgb':
        base = dict(n_estimators=n_estimators, learning_rate=0.03, num_leaves=63, min_child_samples=30, subsample=0.9, colsample_bytree=0.9, random_state=seed, verbosity=-1, n_jobs=-1)
        if task == 'goal': return {**base, 'objective':'poisson'}
        if task == 'outcome': return {**base, 'objective':'multiclass', 'num_class':3}
        if task == 'tail': return {**base, 'objective':'binary'}
    raise ValueError((booster, task))

def make_trial_params(booster, task, trial=None, seed=42, n_estimators=550):
    params = make_default_params(booster, task, seed, n_estimators)
    if trial is None: return params
    if booster == 'cat':
        params['depth'] = trial.suggest_int('depth', 5, 8); params['learning_rate'] = trial.suggest_float('learning_rate', 0.02, 0.06, log=True); params['l2_leaf_reg'] = trial.suggest_float('l2_leaf_reg', 3.0, 8.0)
    elif booster == 'lgb':
        params['num_leaves'] = trial.suggest_int('num_leaves', 31, 95); params['learning_rate'] = trial.suggest_float('learning_rate', 0.02, 0.06, log=True); params['min_child_samples'] = trial.suggest_int('min_child_samples', 20, 60); params['subsample'] = trial.suggest_float('subsample', 0.75, 1.0); params['colsample_bytree'] = trial.suggest_float('colsample_bytree', 0.75, 1.0)
    return params

def prepare_model_frame(df, feature_cols, cat_features):
    X = select_unique_columns(df, feature_cols)
    for c in cat_features:
        if c in X.columns: X[c] = X[c].astype('category')
    for c in X.columns:
        if c not in cat_features: X[c] = pd.to_numeric(X[c], errors='coerce')
    return X

def fit_goal_model(Xtr, ytr, Xva, yva, booster, params, cats):
    if booster == 'cat':
        m = CatBoostRegressor(**params); m.fit(Xtr, ytr, eval_set=(Xva, yva), cat_features=[c for c in cats if c in Xtr.columns], use_best_model=True, verbose=params.get('verbose', MODEL_VERBOSE)); return m
    if booster == 'lgb':
        m = lgb.LGBMRegressor(**params); m.fit(Xtr, ytr, eval_set=[(Xva, yva)], categorical_feature=[c for c in cats if c in Xtr.columns], callbacks=[lgb.early_stopping(100, verbose=LGB_EARLY_STOP_VERBOSE)]); return m
    raise ValueError(booster)

def fit_outcome_model(Xtr, ytr, Xva, yva, booster, params, cats):
    if booster == 'cat':
        m = CatBoostClassifier(**params); m.fit(Xtr, ytr, eval_set=(Xva, yva), cat_features=[c for c in cats if c in Xtr.columns], use_best_model=True, verbose=params.get('verbose', MODEL_VERBOSE)); return m
    if booster == 'lgb':
        m = lgb.LGBMClassifier(**params); m.fit(Xtr, ytr, eval_set=[(Xva, yva)], categorical_feature=[c for c in cats if c in Xtr.columns], callbacks=[lgb.early_stopping(100, verbose=LGB_EARLY_STOP_VERBOSE)]); return m
    raise ValueError(booster)

def fit_tail_model(Xtr, ytr, Xva, yva, booster, params, cats):
    if booster == 'cat':
        m = CatBoostClassifier(**params); m.fit(Xtr, ytr, eval_set=(Xva, yva), cat_features=[c for c in cats if c in Xtr.columns], use_best_model=True, verbose=params.get('verbose', MODEL_VERBOSE)); return m
    if booster == 'lgb':
        m = lgb.LGBMClassifier(**params); m.fit(Xtr, ytr, eval_set=[(Xva, yva)], categorical_feature=[c for c in cats if c in Xtr.columns], callbacks=[lgb.early_stopping(100, verbose=LGB_EARLY_STOP_VERBOSE)]); return m
    raise ValueError(booster)

def predict_reg(m, X): return np.clip(np.asarray(m.predict(X), dtype=float), 0, 30)
def predict_proba3(m, X):
    p = np.asarray(m.predict_proba(X), dtype=float)
    if p.shape[1] < 3:
        out = np.zeros((len(X), 3)) + 1e-6
        for i, cls in enumerate(getattr(m, 'classes_', range(p.shape[1]))): out[:, int(cls)] = p[:, i]
        p = out
    return p / p.sum(axis=1, keepdims=True)
def predict_pos(m, X):
    p = np.asarray(m.predict_proba(X), dtype=float)
    return np.clip(p[:, 1] if p.ndim == 2 and p.shape[1] > 1 else p.ravel(), 1e-6, 1-1e-6)

def filter_regime_train(df, regime):
    if regime == 'full_history': return df.copy()
    if regime == 'modern_cutoff_1990': return df[pd.to_datetime(df['date']) >= pd.Timestamp('1990-01-01')].copy()
    raise ValueError(regime)

def fit_gender_split_goal_models_only(train_df, valid_df, booster_name, regime_name, gender_value, params):
    tr = train_df[train_df['gender'].astype(str) == str(gender_value)].copy(); va = valid_df[valid_df['gender'].astype(str) == str(gender_value)].copy()
    feature_cols, cats = params['feature_cols'], params['cat_features']
    tr = select_unique_columns(tr, ['match_id','gender','tournament','date','y_goal_a','y_goal_b'] + feature_cols); va = select_unique_columns(va, ['match_id','gender','tournament','date','y_goal_a','y_goal_b'] + feature_cols)
    Xtr, Xva = prepare_model_frame(tr, feature_cols, cats), prepare_model_frame(va, feature_cols, cats)
    print(f'[INFO] fit goals-only | booster={booster_name} | regime={regime_name} | gender={gender_value} | n_train={len(tr)} | n_valid={len(va)}')
    return {'booster_name':booster_name, 'regime_name':regime_name, 'gender_value':gender_value, 'feature_cols':feature_cols, 'cat_features':cats,
            'goal_a':fit_goal_model(Xtr, tr['y_goal_a'], Xva, va['y_goal_a'], booster_name, params['goal_params'], cats),
            'goal_b':fit_goal_model(Xtr, tr['y_goal_b'], Xva, va['y_goal_b'], booster_name, params['goal_params'], cats)}

def fit_gender_split_full_heads(train_df, valid_df, booster_name, regime_name, gender_value, params):
    tr = train_df[train_df['gender'].astype(str) == str(gender_value)].copy(); va = valid_df[valid_df['gender'].astype(str) == str(gender_value)].copy()
    feature_cols, cats = params['feature_cols'], params['cat_features']
    targets = ['y_goal_a','y_goal_b','y_outcome','is_team_a_blowout_5plus','is_team_b_blowout_5plus','is_team_a_blowout_7plus','is_team_b_blowout_7plus','is_high_total_6plus']
    tr = select_unique_columns(tr, ['match_id','gender','tournament','date'] + targets + feature_cols); va = select_unique_columns(va, ['match_id','gender','tournament','date'] + targets + feature_cols)
    Xtr, Xva = prepare_model_frame(tr, feature_cols, cats), prepare_model_frame(va, feature_cols, cats)
    print(f'[INFO] fit full heads | booster={booster_name} | regime={regime_name} | gender={gender_value} | n_train={len(tr)} | n_valid={len(va)}')
    return {'booster_name':booster_name, 'regime_name':regime_name, 'gender_value':gender_value, 'feature_cols':feature_cols, 'cat_features':cats,
            'goal_a':fit_goal_model(Xtr, tr['y_goal_a'], Xva, va['y_goal_a'], booster_name, params['goal_params'], cats),
            'goal_b':fit_goal_model(Xtr, tr['y_goal_b'], Xva, va['y_goal_b'], booster_name, params['goal_params'], cats),
            'outcome':fit_outcome_model(Xtr, tr['y_outcome'], Xva, va['y_outcome'], booster_name, params['outcome_params'], cats),
            'tail_a5':fit_tail_model(Xtr, tr['is_team_a_blowout_5plus'], Xva, va['is_team_a_blowout_5plus'], booster_name, params['tail_params'], cats),
            'tail_b5':fit_tail_model(Xtr, tr['is_team_b_blowout_5plus'], Xva, va['is_team_b_blowout_5plus'], booster_name, params['tail_params'], cats),
            'tail_a7':fit_tail_model(Xtr, tr['is_team_a_blowout_7plus'], Xva, va['is_team_a_blowout_7plus'], booster_name, params['tail_params'], cats),
            'tail_b7':fit_tail_model(Xtr, tr['is_team_b_blowout_7plus'], Xva, va['is_team_b_blowout_7plus'], booster_name, params['tail_params'], cats),
            'tail_ht':fit_tail_model(Xtr, tr['is_high_total_6plus'], Xva, va['is_high_total_6plus'], booster_name, params['tail_params'], cats)}

def predict_gender_split_raw_outputs(df, models_by_gender):
    rows = []
    for g, bundle in models_by_gender.items():
        sub = df[df['gender'].astype(str) == str(g)].copy()
        if len(sub) == 0: continue
        X = prepare_model_frame(sub, bundle['feature_cols'], bundle['cat_features'])
        pa, pb = predict_reg(bundle['goal_a'], X), predict_reg(bundle['goal_b'], X)
        out = pd.DataFrame({'match_id':sub['match_id'].values, 'gender':sub['gender'].astype(str).values, 'tournament':sub['tournament'].astype(str).values, 'pred_goal_a_cont':pa, 'pred_goal_b_cont':pb})
        if 'actual_team_a_goals' in sub.columns: out['actual_team_a_goals'] = pd.to_numeric(sub['actual_team_a_goals'], errors='coerce').values
        if 'actual_team_b_goals' in sub.columns: out['actual_team_b_goals'] = pd.to_numeric(sub['actual_team_b_goals'], errors='coerce').values
        if 'outcome' in bundle:
            p = predict_proba3(bundle['outcome'], X)
        else:
            gd = pa - pb; p = np.vstack([1/(1+np.exp(-gd)), np.exp(-np.abs(gd)), 1/(1+np.exp(gd))]).T; p = p / p.sum(axis=1, keepdims=True)
        out['pred_outcome_proba_0'], out['pred_outcome_proba_1'], out['pred_outcome_proba_2'] = p[:,0], p[:,1], p[:,2]
        if 'tail_a5' in bundle:
            out['prob_a_blowout_5plus'] = predict_pos(bundle['tail_a5'], X); out['prob_b_blowout_5plus'] = predict_pos(bundle['tail_b5'], X); out['prob_a_blowout_7plus'] = predict_pos(bundle['tail_a7'], X); out['prob_b_blowout_7plus'] = predict_pos(bundle['tail_b7'], X); out['prob_high_total'] = predict_pos(bundle['tail_ht'], X)
        rows.append(out)
    return pd.concat(rows, ignore_index=True).sort_values('match_id').reset_index(drop=True)


def make_head_feature_config(goal_feature_cols, goal_cat_features, outcome_feature_cols, outcome_cat_features, tail_feature_cols, tail_cat_features):
    return {
        'goal': {'feature_cols': list(goal_feature_cols), 'cat_features': list(goal_cat_features)},
        'outcome': {'feature_cols': list(outcome_feature_cols), 'cat_features': list(outcome_cat_features)},
        'tail': {'feature_cols': list(tail_feature_cols), 'cat_features': list(tail_cat_features)},
    }

HEAD_CONFIGS = {
    'V0_baseline': make_head_feature_config(exp05a_feature_cols, exp05a_cat_features, exp05a_feature_cols, exp05a_cat_features, exp05a_feature_cols, exp05a_cat_features),
    'V1_allhead': make_head_feature_config(exp09a_feature_cols, exp09a_cat_features, exp09a_feature_cols, exp09a_cat_features, exp09a_feature_cols, exp09a_cat_features),
    'V2_outcome_only': make_head_feature_config(exp05a_feature_cols, exp05a_cat_features, exp09a_feature_cols, exp09a_cat_features, exp05a_feature_cols, exp05a_cat_features),
    'V3_outcome_tail': make_head_feature_config(exp05a_feature_cols, exp05a_cat_features, exp09a_feature_cols, exp09a_cat_features, exp09a_feature_cols, exp09a_cat_features),
}

def fit_gender_split_full_heads_with_config(train_df, valid_df, booster_name, regime_name, gender_value, params):
    tr = train_df[train_df['gender'].astype(str) == str(gender_value)].copy()
    va = valid_df[valid_df['gender'].astype(str) == str(gender_value)].copy()

    goal_cols = params['head_feature_config']['goal']['feature_cols']
    goal_cats = params['head_feature_config']['goal']['cat_features']
    outcome_cols = params['head_feature_config']['outcome']['feature_cols']
    outcome_cats = params['head_feature_config']['outcome']['cat_features']
    tail_cols_cfg = params['head_feature_config']['tail']['feature_cols']
    tail_cats = params['head_feature_config']['tail']['cat_features']

    Xtr_goal = prepare_model_frame(tr, goal_cols, goal_cats)
    Xva_goal = prepare_model_frame(va, goal_cols, goal_cats)
    Xtr_out = prepare_model_frame(tr, outcome_cols, outcome_cats)
    Xva_out = prepare_model_frame(va, outcome_cols, outcome_cats)
    Xtr_tail = prepare_model_frame(tr, tail_cols_cfg, tail_cats)
    Xva_tail = prepare_model_frame(va, tail_cols_cfg, tail_cats)

    print(
        f'[INFO] fit full heads by config | booster={booster_name} | regime={regime_name} | '
        f'gender={gender_value} | n_train={len(tr)} | n_valid={len(va)} | '
        f'goal_features={len(goal_cols)} | outcome_features={len(outcome_cols)} | tail_features={len(tail_cols_cfg)}',
        flush=True
    )

    return {
        'booster_name': booster_name,
        'regime_name': regime_name,
        'gender_value': gender_value,
        'head_feature_config': copy.deepcopy(params['head_feature_config']),
        'goal_a': fit_goal_model(Xtr_goal, tr['y_goal_a'], Xva_goal, va['y_goal_a'], booster_name, params['goal_params'], goal_cats),
        'goal_b': fit_goal_model(Xtr_goal, tr['y_goal_b'], Xva_goal, va['y_goal_b'], booster_name, params['goal_params'], goal_cats),
        'outcome': fit_outcome_model(Xtr_out, tr['y_outcome'], Xva_out, va['y_outcome'], booster_name, params['outcome_params'], outcome_cats),
        'tail_a5': fit_tail_model(Xtr_tail, tr['is_team_a_blowout_5plus'], Xva_tail, va['is_team_a_blowout_5plus'], booster_name, params['tail_params'], tail_cats),
        'tail_b5': fit_tail_model(Xtr_tail, tr['is_team_b_blowout_5plus'], Xva_tail, va['is_team_b_blowout_5plus'], booster_name, params['tail_params'], tail_cats),
        'tail_a7': fit_tail_model(Xtr_tail, tr['is_team_a_blowout_7plus'], Xva_tail, va['is_team_a_blowout_7plus'], booster_name, params['tail_params'], tail_cats),
        'tail_b7': fit_tail_model(Xtr_tail, tr['is_team_b_blowout_7plus'], Xva_tail, va['is_team_b_blowout_7plus'], booster_name, params['tail_params'], tail_cats),
        'tail_ht': fit_tail_model(Xtr_tail, tr['is_high_total_6plus'], Xva_tail, va['is_high_total_6plus'], booster_name, params['tail_params'], tail_cats),
    }

def predict_gender_split_raw_outputs_with_config(df, models_by_gender):
    rows = []
    for g, bundle in models_by_gender.items():
        sub = df[df['gender'].astype(str) == str(g)].copy()
        if len(sub) == 0:
            continue
        cfg = bundle['head_feature_config']
        X_goal = prepare_model_frame(sub, cfg['goal']['feature_cols'], cfg['goal']['cat_features'])
        X_out = prepare_model_frame(sub, cfg['outcome']['feature_cols'], cfg['outcome']['cat_features'])
        X_tail = prepare_model_frame(sub, cfg['tail']['feature_cols'], cfg['tail']['cat_features'])

        pa = predict_reg(bundle['goal_a'], X_goal)
        pb = predict_reg(bundle['goal_b'], X_goal)
        out = pd.DataFrame({
            'match_id': sub['match_id'].values,
            'gender': sub['gender'].astype(str).values,
            'tournament': sub['tournament'].astype(str).values,
            'pred_goal_a_cont': pa,
            'pred_goal_b_cont': pb,
        })
        if 'tournament_structure' in sub.columns:
            out['tournament_structure'] = sub['tournament_structure'].astype(str).values
        if 'actual_team_a_goals' in sub.columns:
            out['actual_team_a_goals'] = pd.to_numeric(sub['actual_team_a_goals'], errors='coerce').values
        if 'actual_team_b_goals' in sub.columns:
            out['actual_team_b_goals'] = pd.to_numeric(sub['actual_team_b_goals'], errors='coerce').values

        p = predict_proba3(bundle['outcome'], X_out)
        out['pred_outcome_proba_0'], out['pred_outcome_proba_1'], out['pred_outcome_proba_2'] = p[:, 0], p[:, 1], p[:, 2]
        out['prob_a_blowout_5plus'] = predict_pos(bundle['tail_a5'], X_tail)
        out['prob_b_blowout_5plus'] = predict_pos(bundle['tail_b5'], X_tail)
        out['prob_a_blowout_7plus'] = predict_pos(bundle['tail_a7'], X_tail)
        out['prob_b_blowout_7plus'] = predict_pos(bundle['tail_b7'], X_tail)
        out['prob_high_total'] = predict_pos(bundle['tail_ht'], X_tail)
        rows.append(out)

    return pd.concat(rows, ignore_index=True).sort_values('match_id').reset_index(drop=True)

def train_final_gender_models_by_head_config(train_df, valid_df, final_selected, head_feature_config, n_estimators=1200):
    models = {}
    for gender, cfg in final_selected.items():
        booster, regime = cfg['booster'], cfg['regime']
        tr = filter_regime_train(train_df, regime)
        models[gender] = fit_gender_split_full_heads_with_config(
            tr,
            valid_df,
            booster,
            regime,
            gender,
            {
                'head_feature_config': head_feature_config,
                'goal_params': make_default_params(booster, 'goal', SEED, n_estimators),
                'outcome_params': make_default_params(booster, 'outcome', SEED, n_estimators),
                'tail_params': make_default_params(booster, 'tail', SEED, n_estimators),
            },
        )
    return models


def build_scoreline_prior(goal_a, goal_b, max_goals, alpha=1.0):
    counts = np.zeros((max_goals+1, max_goals+1), dtype=float) + alpha
    for a,b in zip(goal_a, goal_b): counts[int(np.clip(a,0,max_goals)), int(np.clip(b,0,max_goals))] += 1
    counts /= counts.sum()
    return {(a,b):float(counts[a,b]) for a in range(max_goals+1) for b in range(max_goals+1)}
scoreline_prior_lookup = {mg:build_scoreline_prior(y_train_fold['y_goal_a'], y_train_fold['y_goal_b'], mg, 1.0) for mg in [7,9,11]}

def decode_anchor_batch_from_raw(raw_df, params, prior, eps=1e-9):
    max_goals = int(params['MAX_GOALS']); cand = [(a,b) for a in range(max_goals+1) for b in range(max_goals+1)]
    ca, cb = np.array([x[0] for x in cand]), np.array([x[1] for x in cand]); co = np.array([_outcome(a,b) for a,b in cand])
    prior_cost = np.array([-np.log(prior.get((a,b), eps)+eps) for a,b in cand])
    goal_a_anchor_col = 'decode_goal_a_anchor' if 'decode_goal_a_anchor' in raw_df.columns else 'pred_goal_a_cont'
    goal_b_anchor_col = 'decode_goal_b_anchor' if 'decode_goal_b_anchor' in raw_df.columns else 'pred_goal_b_cont'
    pred_a = raw_df[goal_a_anchor_col].to_numpy(float)[:,None]; pred_b = raw_df[goal_b_anchor_col].to_numpy(float)[:,None]
    p = raw_df[['pred_outcome_proba_0','pred_outcome_proba_1','pred_outcome_proba_2']].to_numpy(float)
    cost = params['w_direct']*(np.abs(ca-pred_a)+np.abs(cb-pred_b)) + params['w_outcome']*(-np.log(p[:,co]+eps)) + params['w_prior']*prior_cost
    idx = np.argmin(cost, axis=1)
    return pd.DataFrame({'match_id':raw_df['match_id'].values, 'pred_team_a_goals':ca[idx].astype(int), 'pred_team_b_goals':cb[idx].astype(int)})

def decode_directional_tail_batch(raw_df, tail_prob_df, params, prior, eps=1e-9):
    tail_cols = ['prob_a_blowout_5plus','prob_b_blowout_5plus','prob_a_blowout_7plus','prob_b_blowout_7plus','prob_high_total']
    raw_clean = raw_df.drop(columns=tail_cols, errors='ignore')
    merged = raw_clean.merge(tail_prob_df[['match_id']+tail_cols], on='match_id', how='left', validate='one_to_one')
    assert not any(c.endswith('_x') or c.endswith('_y') for c in merged.columns), 'suffix leak in tail merge'
    max_goals = int(params['MAX_GOALS']); cand = [(a,b) for a in range(max_goals+1) for b in range(max_goals+1)]
    ca, cb = np.array([x[0] for x in cand]), np.array([x[1] for x in cand]); co = np.array([_outcome(a,b) for a,b in cand])
    prior_cost = np.array([-np.log(prior.get((a,b), eps)+eps) for a,b in cand])
    goal_a_anchor_col = 'decode_goal_a_anchor' if 'decode_goal_a_anchor' in merged.columns else 'pred_goal_a_cont'
    goal_b_anchor_col = 'decode_goal_b_anchor' if 'decode_goal_b_anchor' in merged.columns else 'pred_goal_b_cont'
    pred_a = merged[goal_a_anchor_col].to_numpy(float)[:,None]; pred_b = merged[goal_b_anchor_col].to_numpy(float)[:,None]
    p = merged[['pred_outcome_proba_0','pred_outcome_proba_1','pred_outcome_proba_2']].to_numpy(float)
    cost = params['w_direct']*(np.abs(ca-pred_a)+np.abs(cb-pred_b)) + params['w_outcome']*(-np.log(p[:,co]+eps)) + params['w_prior']*prior_cost
    for c in tail_cols: merged[c] = pd.to_numeric(merged[c], errors='coerce').fillna(eps).clip(eps,1-eps)
    masks = {'w_a5':(ca-cb)>=5, 'w_b5':(cb-ca)>=5, 'w_a7':(ca-cb)>=7, 'w_b7':(cb-ca)>=7, 'w_ht':(ca+cb)>=6}
    probs = {'w_a5':'prob_a_blowout_5plus', 'w_b5':'prob_b_blowout_5plus', 'w_a7':'prob_a_blowout_7plus', 'w_b7':'prob_b_blowout_7plus', 'w_ht':'prob_high_total'}
    for w, mask in masks.items(): cost -= float(params.get(w,0))*mask*np.log(merged[probs[w]].to_numpy(float)[:,None]+eps)
    idx = np.argmin(cost, axis=1)
    return pd.DataFrame({'match_id':merged['match_id'].values, 'pred_team_a_goals':ca[idx].astype(int), 'pred_team_b_goals':cb[idx].astype(int)})

def tune_decoder_from_raw_outputs(raw_df, grid, priors):
    rows, cache = [], []
    for combo in iter_progress(list(product(*[grid[k] for k in grid])), desc='tune_decoder'):
        params = {k:v for k,v in zip(grid.keys(), combo)}; params['MAX_GOALS'] = int(params['MAX_GOALS'])
        pred = decode_anchor_batch_from_raw(raw_df, params, priors[params['MAX_GOALS']])
        score = awmae_score(raw_df['actual_team_a_goals'], raw_df['actual_team_b_goals'], pred['pred_team_a_goals'], pred['pred_team_b_goals'], raw_df['tournament'])
        rows.append({**params, 'valid_awmae':score}); cache.append((score, params, pred))
    best = sorted(cache, key=lambda x:x[0])[0]
    return pd.DataFrame(rows).sort_values('valid_awmae').reset_index(drop=True), best[1], best[2]

def tune_tail_aware_decoder_from_raw_outputs(raw_df, tail_prob_df, grid, priors, top_n=3):
    rows, cache = [], []
    raw_clean = raw_df.drop(columns=['prob_a_blowout_5plus','prob_b_blowout_5plus','prob_a_blowout_7plus','prob_b_blowout_7plus','prob_high_total'], errors='ignore')
    for combo in iter_progress(list(product(*[grid[k] for k in grid])), desc='tune_tail_decoder'):
        params = {k:v for k,v in zip(grid.keys(), combo)}; params['MAX_GOALS'] = int(params['MAX_GOALS'])
        pred = decode_directional_tail_batch(raw_clean, tail_prob_df, params, priors[params['MAX_GOALS']])
        score = awmae_score(raw_df['actual_team_a_goals'], raw_df['actual_team_b_goals'], pred['pred_team_a_goals'], pred['pred_team_b_goals'], raw_df['tournament'])
        rows.append({**params, 'valid_awmae':score}); cache.append((score, params, pred))
    best = sorted(cache, key=lambda x:x[0])[0]
    return pd.DataFrame(rows).sort_values('valid_awmae').reset_index(drop=True), best[1], best[2]


train_sup_full = train_feature_full.merge(y_train_fold, on='match_id', how='left', validate='one_to_one')
valid_sup_full = valid_feature_full.merge(y_valid_fold, on='match_id', how='left', validate='one_to_one')

# Anchor validation decode fix from source.
# valid_feature_full already has gender and tournament, so merge only actual goals.
valid_decode_df = valid_feature_full.copy()
valid_decode_df = valid_decode_df.drop(columns=['actual_team_a_goals', 'actual_team_b_goals'], errors='ignore')
valid_target_df = valid_fold_base[['match_id', 'team_a_goals', 'team_b_goals']].rename(columns={
    'team_a_goals': 'actual_team_a_goals',
    'team_b_goals': 'actual_team_b_goals',
})
valid_decode_df = valid_decode_df.merge(valid_target_df, on='match_id', how='left', validate='one_to_one')
required_cols = ['match_id', 'gender', 'tournament', 'actual_team_a_goals', 'actual_team_b_goals']
missing_cols = [c for c in required_cols if c not in valid_decode_df.columns]
assert len(missing_cols) == 0, f'valid_decode_df missing columns: {missing_cols}'
suffix_cols = [c for c in valid_decode_df.columns if c.endswith('_x') or c.endswith('_y')]
assert len(suffix_cols) == 0, f'valid_decode_df has bad suffix columns: {suffix_cols}'
assert valid_decode_df['actual_team_a_goals'].notna().all()
assert valid_decode_df['actual_team_b_goals'].notna().all()
log_check('Anchor validation decode has gender', 'gender' in valid_decode_df.columns)
log_check('Anchor validation decode has no suffix columns', len(suffix_cols) == 0)


def run_screening_unit(train_df, valid_df, booster, regime, gender, n_trials=10):
    train_regime = filter_regime_train(train_df, regime)
    def objective(trial):
        params = make_trial_params(booster, 'goal', trial, SEED+trial.number, 550)
        bundle = fit_gender_split_goal_models_only(train_regime, valid_df, booster, regime, gender, {'feature_cols':student_feature_cols, 'cat_features':student_cat_features, 'goal_params':params})
        vg = valid_df[valid_df['gender'].astype(str)==str(gender)].rename(columns={'y_goal_a':'actual_team_a_goals','y_goal_b':'actual_team_b_goals'})
        va_decode = select_unique_columns(vg, ['match_id','gender','tournament','actual_team_a_goals','actual_team_b_goals'] + student_feature_cols)
        raw = predict_gender_split_raw_outputs(va_decode, {gender:bundle})
        pa, pb = np.clip(np.rint(raw['pred_goal_a_cont']),0,9).astype(int), np.clip(np.rint(raw['pred_goal_b_cont']),0,9).astype(int)
        return awmae_score(raw['actual_team_a_goals'], raw['actual_team_b_goals'], pa, pb, raw['tournament'])
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return {'regime':regime, 'gender':gender, 'booster':booster, 'screen_valid_awmae':float(study.best_value), 'best_params':study.best_params, 'n_trials':n_trials, 'status':'ok'}

screen_rows = []
for regime in CUTOFF_REGIMES:
    for gender in sorted(train_fold_base['gender'].astype(str).unique()):
        for booster in BOOSTERS_TO_RUN:
            try: screen_rows.append(run_screening_unit(train_sup_full, valid_sup_full, booster, regime, gender, SCREEN_N_TRIALS))
            except Exception as e: screen_rows.append({'regime':regime,'gender':gender,'booster':booster,'screen_valid_awmae':np.inf,'best_params':{},'n_trials':SCREEN_N_TRIALS,'status':f'failed: {repr(e)}'})
screening_results = pd.DataFrame(screen_rows).sort_values(['gender','screen_valid_awmae']).reset_index(drop=True)
screening_results.to_csv(SUM_DIR/'booster_screening_results.csv', index=False)
display(screening_results)
with open(SUM_DIR/'xgb_policy.json','w') as f: json.dump({'xgb_policy':XGB_POLICY, 'reason':'skipped for clean fix-v2 due prior unseen categorical error'}, f, indent=2)

def make_temporal_inner_splits(df, n_splits=2):
    d = df.sort_values(['date','match_id']).reset_index(drop=True); n=len(d); out=[]
    for frac in [0.60, 0.70][:n_splits]:
        cut=int(n*frac); vs=max(1,int(n*0.15)); tr=d.iloc[:cut].copy(); va=d.iloc[cut:min(n,cut+vs)].copy()
        if len(tr) and len(va): out.append((tr,va))
    return out

def run_refine_for_candidate(train_df, booster, regime, gender, n_trials=20):
    train_g = filter_regime_train(train_df, regime)
    train_g = train_g[train_g['gender'].astype(str)==str(gender)].copy()
    splits = make_temporal_inner_splits(train_g, 2)
    def objective(trial):
        scores=[]; params = make_trial_params(booster, 'goal', trial, SEED+1000+trial.number, 850)
        for tr, va in splits:
            bundle = fit_gender_split_goal_models_only(tr, va, booster, regime, gender, {'feature_cols':student_feature_cols, 'cat_features':student_cat_features, 'goal_params':params})
            va_decode = select_unique_columns(va, ['match_id','gender','tournament','y_goal_a','y_goal_b'] + student_feature_cols).rename(columns={'y_goal_a':'actual_team_a_goals','y_goal_b':'actual_team_b_goals'})
            assert_no_duplicate_columns(va_decode, 'refine va_decode')
            raw = predict_gender_split_raw_outputs(va_decode, {gender:bundle})
            pa, pb = np.clip(np.rint(raw['pred_goal_a_cont']),0,9).astype(int), np.clip(np.rint(raw['pred_goal_b_cont']),0,9).astype(int)
            scores.append(awmae_score(raw['actual_team_a_goals'], raw['actual_team_b_goals'], pa, pb, raw['tournament']))
        return float(np.mean(scores))
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED+99))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return {'regime':regime, 'gender':gender, 'booster':booster, 'refine_valid_awmae':float(study.best_value), 'best_params':study.best_params, 'n_trials':n_trials, 'status':'ok', 'source':'refine'}

refine_candidates = screening_results[screening_results['status'].eq('ok')].sort_values('screen_valid_awmae').groupby('gender', as_index=False).head(1).copy()
refine_rows=[]
for _, r in refine_candidates.iterrows():
    try: refine_rows.append(run_refine_for_candidate(train_sup_full, r['booster'], r['regime'], r['gender'], REFINE_N_TRIALS))
    except Exception as e: refine_rows.append({'regime':r['regime'], 'gender':r['gender'], 'booster':r['booster'], 'refine_valid_awmae':np.inf, 'best_params':{}, 'n_trials':REFINE_N_TRIALS, 'status':f'failed: {repr(e)}', 'source':'refine_failed'})
refine_results = pd.DataFrame(refine_rows).sort_values(['gender','refine_valid_awmae']).reset_index(drop=True)
refine_results.to_csv(SUM_DIR/'booster_refine_results.csv', index=False)
display(refine_results)

final_selected = {}
for gender in sorted(train_fold_base['gender'].astype(str).unique()):
    rr = refine_results[(refine_results['gender'].astype(str)==str(gender)) & refine_results['status'].eq('ok')]
    if len(rr):
        best = rr.sort_values('refine_valid_awmae').iloc[0]
        final_selected[str(gender)] = {'booster':str(best['booster']), 'regime':str(best['regime']), 'source':'refine', 'valid_awmae':float(best['refine_valid_awmae'])}
    else:
        ss = screening_results[(screening_results['gender'].astype(str)==str(gender)) & screening_results['status'].eq('ok')].sort_values('screen_valid_awmae')
        best = ss.iloc[0]
        final_selected[str(gender)] = {'booster':str(best['booster']), 'regime':str(best['regime']), 'source':'screen_fallback', 'valid_awmae':float(best['screen_valid_awmae']), 'fallback_reason':'refine_failed'}
with open(SUM_DIR/'final_selected_boosters.json','w',encoding='utf-8') as f: json.dump(final_selected, f, indent=2, ensure_ascii=False)
final_selected_df = pd.DataFrame([{'gender': g, **cfg} for g, cfg in final_selected.items()])
display(final_selected_df)


def prediction_detail_table(pred_df, variant_name):
    df = pred_df.copy()
    df['variant_name'] = variant_name
    df['base_mae'] = (
        (df['actual_team_a_goals'] - df['pred_team_a_goals']).abs()
        + (df['actual_team_b_goals'] - df['pred_team_b_goals']).abs()
    ) / 2
    df['exact_hit'] = (
        (df['actual_team_a_goals'].astype(int) == df['pred_team_a_goals'].astype(int))
        & (df['actual_team_b_goals'].astype(int) == df['pred_team_b_goals'].astype(int))
    ).astype(int)
    df['true_outcome'] = [_outcome(a, b) for a, b in zip(df['actual_team_a_goals'], df['actual_team_b_goals'])]
    df['pred_outcome'] = [_outcome(a, b) for a, b in zip(df['pred_team_a_goals'], df['pred_team_b_goals'])]
    df['outcome_hit'] = (df['true_outcome'] == df['pred_outcome']).astype(int)
    df['true_gd'] = df['actual_team_a_goals'] - df['actual_team_b_goals']
    df['pred_gd'] = df['pred_team_a_goals'] - df['pred_team_b_goals']
    df['gd_hit'] = (df['true_gd'] == df['pred_gd']).astype(int)
    df['true_total'] = df['actual_team_a_goals'] + df['actual_team_b_goals']
    df['pred_total'] = df['pred_team_a_goals'] + df['pred_team_b_goals']
    df['abs_true_gd'] = df['true_gd'].abs()
    df['is_blowout_5plus'] = (df['abs_true_gd'] >= 5).astype(int)
    df['is_blowout_7plus'] = (df['abs_true_gd'] >= 7).astype(int)
    df['is_high_total_6plus'] = (df['true_total'] >= 6).astype(int)
    df['scoreline'] = df['pred_team_a_goals'].astype(int).astype(str) + '-' + df['pred_team_b_goals'].astype(int).astype(str)
    df['official_loss'] = [
        official_match_loss(a, b, pa, pb)
        for a, b, pa, pb in zip(df['actual_team_a_goals'], df['actual_team_b_goals'], df['pred_team_a_goals'], df['pred_team_b_goals'])
    ]
    df['tournament_weight'] = [get_tournament_weight(t) for t in df['tournament']]
    return df

def summarize_prediction_detail(detail_df, variant_name):
    score_counts = detail_df['scoreline'].value_counts(normalize=True)
    top3_share = float(score_counts.head(3).sum()) if len(score_counts) else 0.0
    return {
        'variant_name': variant_name,
        'awmae': awmae_score(
            detail_df['actual_team_a_goals'], detail_df['actual_team_b_goals'],
            detail_df['pred_team_a_goals'], detail_df['pred_team_b_goals'],
            detail_df['tournament'],
        ),
        'base_mae': float(detail_df['base_mae'].mean()),
        'exact_rate': float(detail_df['exact_hit'].mean()),
        'outcome_rate': float(detail_df['outcome_hit'].mean()),
        'gd_rate': float(detail_df['gd_hit'].mean()),
        'team_bias': float((detail_df['pred_team_a_goals'] - detail_df['actual_team_a_goals']).mean()),
        'opp_bias': float((detail_df['pred_team_b_goals'] - detail_df['actual_team_b_goals']).mean()),
        'total_goal_bias': float((detail_df['pred_total'] - detail_df['true_total']).mean()),
        'mean_pred_total': float(detail_df['pred_total'].mean()),
        'top3_scoreline_share': top3_share,
    }

def compute_domain_error(detail_df, group_col):
    if group_col not in detail_df.columns:
        return pd.DataFrame()
    rows = []
    for key, g in detail_df.groupby(group_col, dropna=False):
        rows.append({
            group_col: key,
            'n': len(g),
            'awmae': awmae_score(g['actual_team_a_goals'], g['actual_team_b_goals'], g['pred_team_a_goals'], g['pred_team_b_goals'], g['tournament']),
            'base_mae': float(g['base_mae'].mean()),
            'exact_rate': float(g['exact_hit'].mean()),
            'outcome_rate': float(g['outcome_hit'].mean()),
            'gd_rate': float(g['gd_hit'].mean()),
        })
    return pd.DataFrame(rows).sort_values('awmae').reset_index(drop=True)

def scoreline_distribution(detail_df):
    rows = []
    for variant, g in detail_df.groupby('variant_name'):
        vc = g['scoreline'].value_counts().reset_index()
        vc.columns = ['scoreline', 'count']
        vc['share'] = vc['count'] / len(g)
        vc['variant_name'] = variant
        rows.append(vc[['variant_name', 'scoreline', 'count', 'share']])
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

TAIL_COLS = ['prob_a_blowout_5plus','prob_b_blowout_5plus','prob_a_blowout_7plus','prob_b_blowout_7plus','prob_high_total']
TAIL_GRID = {
    'MAX_GOALS': [7, 9],
    'w_direct': [0.75, 1.0],
    'w_outcome': [1.5, 2.0],
    'w_prior': [0.2, 0.5],
    'w_a5': [0.0, 0.5],
    'w_b5': [0.0, 0.5],
    'w_a7': [0.0, 0.5],
    'w_b7': [0.0, 0.5],
    'w_ht': [0.0, 0.5],
}

def run_exp09c_variant(variant_name, head_feature_config, save_name):
    log_section(f'EXP09C variant: {variant_name}')
    models = train_final_gender_models_by_head_config(train_sup_full, valid_sup_full, final_selected, head_feature_config, 1200)
    raw_valid = predict_gender_split_raw_outputs_with_config(valid_decode_df, models)
    tail_prob_valid = raw_valid[['match_id'] + TAIL_COLS].copy()
    decoder_grid, best_decoder, pred = tune_tail_aware_decoder_from_raw_outputs(
        raw_valid,
        tail_prob_valid,
        TAIL_GRID,
        scoreline_prior_lookup,
        top_n=3,
    )
    pred_match = raw_valid.merge(pred, on='match_id', how='left', validate='one_to_one')
    detail = prediction_detail_table(pred_match, variant_name)
    pred_path = PRED_DIR / f'valid_pred_match_{save_name}.csv'
    pred_match.to_csv(pred_path, index=False)
    log_saved(pred_path)
    return {
        'variant_name': variant_name,
        'models': models,
        'raw_valid': raw_valid,
        'pred_match': pred_match,
        'detail': detail,
        'decoder_grid': decoder_grid,
        'best_decoder': best_decoder,
        'summary': summarize_prediction_detail(detail, variant_name),
    }

VARIANT_RESULTS = {}


 # 10. Train EXP09D Base dan Repair Outcome Sources



 Bagian ini melatih sumber model seperti EXP09D. Goal dan tail tetap baseline EXP05A,

 sedangkan outcome punya dua sumber: baseline EXP05A dan repair EXP05A+EXP09A.

In [ ]:
log_section('Train baseline and repair outcome sources')

# Baseline models: goal, outcome, and tail all use EXP05A feature set.
# Repair models: goal/tail still follow EXP05A in the V2-style config, while outcome uses EXP05A + EXP09A features.
# For gating, raw_base is the anchor because goal and tail must remain baseline.
BASE_HEAD_CONFIG = HEAD_CONFIGS['V0_baseline']
REPAIR_HEAD_CONFIG = HEAD_CONFIGS['V2_outcome_only']

base_models = train_final_gender_models_by_head_config(
    train_sup_full,
    valid_sup_full,
    final_selected,
    BASE_HEAD_CONFIG,
    1200,
)
repair_models = train_final_gender_models_by_head_config(
    train_sup_full,
    valid_sup_full,
    final_selected,
    REPAIR_HEAD_CONFIG,
    1200,
)

head_feature_config_df = pd.DataFrame([
    {'head': 'goal_a', 'source_for_gate': 'baseline', 'feature_set': 'EXP05A', 'n_features': len(BASE_HEAD_CONFIG['goal']['feature_cols']), 'n_cat_features': len(BASE_HEAD_CONFIG['goal']['cat_features'])},
    {'head': 'goal_b', 'source_for_gate': 'baseline', 'feature_set': 'EXP05A', 'n_features': len(BASE_HEAD_CONFIG['goal']['feature_cols']), 'n_cat_features': len(BASE_HEAD_CONFIG['goal']['cat_features'])},
    {'head': 'tail', 'source_for_gate': 'baseline', 'feature_set': 'EXP05A', 'n_features': len(BASE_HEAD_CONFIG['tail']['feature_cols']), 'n_cat_features': len(BASE_HEAD_CONFIG['tail']['cat_features'])},
    {'head': 'outcome_base', 'source_for_gate': 'baseline', 'feature_set': 'EXP05A', 'n_features': len(BASE_HEAD_CONFIG['outcome']['feature_cols']), 'n_cat_features': len(BASE_HEAD_CONFIG['outcome']['cat_features'])},
    {'head': 'outcome_repair', 'source_for_gate': 'repair', 'feature_set': 'EXP05A_PLUS_EXP09A', 'n_features': len(REPAIR_HEAD_CONFIG['outcome']['feature_cols']), 'n_cat_features': len(REPAIR_HEAD_CONFIG['outcome']['cat_features'])},
])
head_feature_config_df.to_csv(SUM_DIR / 'head_feature_config.csv', index=False)
display(head_feature_config_df)
log_saved(SUM_DIR / 'head_feature_config.csv')


 # 11. Build Raw Prediction EXP09D Gate



 Bagian ini menghasilkan raw prediction validation. raw_base menjadi anchor untuk goal/tail,

 sedangkan raw_repair hanya dipakai mengambil outcome probability repair sebelum tournament gate.

In [ ]:
log_section('Build raw prediction for tournament gate')

raw_base_valid = predict_gender_split_raw_outputs_with_config(valid_decode_df, base_models)
raw_repair_valid = predict_gender_split_raw_outputs_with_config(valid_decode_df, repair_models)

prob_cols = ['pred_outcome_proba_0', 'pred_outcome_proba_1', 'pred_outcome_proba_2']
assert raw_base_valid['match_id'].is_unique, 'raw_base_valid match_id is not unique'
assert raw_repair_valid['match_id'].is_unique, 'raw_repair_valid match_id is not unique'
assert set(raw_base_valid['match_id']) == set(raw_repair_valid['match_id']), 'base/repair raw predictions have different match ids'

raw_base_valid.to_csv(PRED_DIR / 'valid_raw_outcome_base.csv', index=False)
raw_repair_valid.to_csv(PRED_DIR / 'valid_raw_outcome_repair.csv', index=False)
log_saved(PRED_DIR / 'valid_raw_outcome_base.csv')
log_saved(PRED_DIR / 'valid_raw_outcome_repair.csv')

base_vs_repair_proba_delta = pd.DataFrame({
    'metric': ['mean_abs_delta_p0', 'mean_abs_delta_p1', 'mean_abs_delta_p2'],
    'value': [
        float(np.mean(np.abs(raw_base_valid['pred_outcome_proba_0'].values - raw_repair_valid['pred_outcome_proba_0'].values))),
        float(np.mean(np.abs(raw_base_valid['pred_outcome_proba_1'].values - raw_repair_valid['pred_outcome_proba_1'].values))),
        float(np.mean(np.abs(raw_base_valid['pred_outcome_proba_2'].values - raw_repair_valid['pred_outcome_proba_2'].values))),
    ],
})
display(base_vs_repair_proba_delta)



 # 12. EXP09D Tournament-Gated Outcome Probability



 Bagian ini mempertahankan tournament gate EXP09D. Gate ini menentukan seberapa besar

 probability outcome repair dipakai berdasarkan tournament_structure. EXP11B memakai gate terpilih

 sebagai baseline sebelum menguji anchor skor.

In [ ]:
def get_gate_alpha(tournament_structure: str, gate_config: dict, default_alpha: float = 0.0) -> float:
    ts = str(tournament_structure)
    return float(gate_config.get(ts, default_alpha))


def apply_tournament_gated_outcome(raw_base, raw_repair, gate_config, default_alpha=0.0):
    """
    Apply tournament-gated outcome probability only.
    Goal and tail signals are preserved from raw_base.
    """
    prob_cols = ['pred_outcome_proba_0', 'pred_outcome_proba_1', 'pred_outcome_proba_2']
    if 'tournament_structure' not in raw_base.columns:
        raise KeyError('raw_base must contain tournament_structure for EXP09D gate')

    base_cols = ['match_id', 'tournament_structure'] + prob_cols
    repair_cols = ['match_id'] + prob_cols
    base = raw_base[base_cols].copy()
    repair = raw_repair[repair_cols].copy()

    merged = base.merge(
        repair,
        on='match_id',
        how='inner',
        suffixes=('_base', '_repair'),
        validate='one_to_one',
    )
    assert len(merged) == len(raw_base), 'Gate merge changed row count'

    alpha = merged['tournament_structure'].map(
        lambda x: get_gate_alpha(x, gate_config, default_alpha)
    ).astype(float).to_numpy()
    alpha = np.clip(alpha, 0.0, 1.0)

    p_base = merged[[c + '_base' for c in prob_cols]].to_numpy(float)
    p_repair = merged[[c + '_repair' for c in prob_cols]].to_numpy(float)
    p = (1 - alpha[:, None]) * p_base + alpha[:, None] * p_repair
    p = np.clip(p, 1e-9, 1.0)
    p = p / p.sum(axis=1, keepdims=True)

    out = raw_base.copy().drop(columns=prob_cols, errors='ignore')
    out = out.merge(
        pd.DataFrame({
            'match_id': merged['match_id'].values,
            'gate_alpha': alpha,
            'pred_outcome_proba_0': p[:, 0],
            'pred_outcome_proba_1': p[:, 1],
            'pred_outcome_proba_2': p[:, 2],
        }),
        on='match_id',
        how='left',
        validate='one_to_one',
    )
    assert out[prob_cols].notna().all().all(), 'Gated outcome probability has NaN'
    assert out['gate_alpha'].notna().all(), 'Gate alpha has NaN'
    return out


GATE_CONFIGS = {
    'V0_baseline_outcome': {
        'default_alpha': 0.0,
        'gate_map': {},
        'description': 'All tournament structures use baseline EXP05A outcome probability.',
    },
    'V1_full_repair': {
        'default_alpha': 1.0,
        'gate_map': {},
        'description': 'All tournament structures use repair EXP09C outcome probability.',
    },
    'V2_hard_helpful_gate': {
        'default_alpha': 0.0,
        'gate_map': {
            'friendly': 1.0,
            'continental_group': 1.0,
            'regional_games': 1.0,
            'world_cup_group': 1.0,
        },
        'description': 'Repair is enabled only for tournament structures that looked helpful in EXP09C validation.',
    },
    'V3_conservative_soft_gate': {
        'default_alpha': 0.5,
        'gate_map': {
            'friendly': 1.0,
            'continental_group': 1.0,
            'regional_games': 1.0,
            'world_cup_group': 1.0,
            'qualifier': 0.25,
            'other_competitive': 0.25,
            'continental_knockout': 0.5,
            'world_cup_knockout': 0.5,
            'nations_league_group': 0.5,
            'nations_league_knockout': 0.5,
            'unknown': 0.5,
        },
        'description': 'A conservative soft gate: helpful structures get full repair, risky structures get weak repair.',
    },
}

with open(SUM_DIR / 'gate_variant_config.json', 'w', encoding='utf-8') as f:
    json.dump(GATE_CONFIGS, f, indent=2, ensure_ascii=False)

map_rows = []
known_structures = sorted(set(raw_base_valid.get('tournament_structure', pd.Series(dtype=str)).astype(str).unique().tolist()))
for variant_name, cfg in GATE_CONFIGS.items():
    covered = sorted(set(known_structures) | set(cfg['gate_map'].keys()))
    for ts in covered:
        alpha_value = get_gate_alpha(ts, cfg['gate_map'], cfg['default_alpha'])
        map_rows.append({
            'variant_name': variant_name,
            'tournament_structure': ts,
            'alpha': alpha_value,
            'source': 'repair' if alpha_value == 1.0 else ('baseline' if alpha_value == 0.0 else 'blend'),
            'default_alpha': cfg['default_alpha'],
            'description': cfg['description'],
        })

tournament_gate_map_df = pd.DataFrame(map_rows)
tournament_gate_map_df.to_csv(SUM_DIR / 'tournament_gate_map.csv', index=False)
display(tournament_gate_map_df.head(80))
log_saved(SUM_DIR / 'gate_variant_config.json')
log_saved(SUM_DIR / 'tournament_gate_map.csv')



 # 13. EXP09D Gate Selection sebagai Baseline EXP11B



 Bagian ini mereplikasi selection gate EXP09D pada validation. Setiap gate variant mendapat decoder

 yang ditune sendiri supaya V0, V1, V2, dan V3 dibandingkan secara fair. Hasil gate terpilih menjadi

 raw_gate baseline yang akan dipakai oleh eksperimen Poisson median dan shrinkage.

In [ ]:
def compute_domain_error_per_variant(detail_df, group_col):
    if group_col not in detail_df.columns:
        return pd.DataFrame()
    rows = []
    for (variant, key), g in detail_df.groupby(['variant_name', group_col], dropna=False):
        rows.append({
            'variant_name': variant,
            group_col: key,
            'n': len(g),
            'awmae': awmae_score(g['actual_team_a_goals'], g['actual_team_b_goals'], g['pred_team_a_goals'], g['pred_team_b_goals'], g['tournament']),
            'base_mae': float(g['base_mae'].mean()),
            'exact_rate': float(g['exact_hit'].mean()),
            'outcome_rate': float(g['outcome_hit'].mean()),
            'gd_rate': float(g['gd_hit'].mean()),
        })
    return pd.DataFrame(rows).sort_values(['variant_name', 'awmae']).reset_index(drop=True)


def evaluate_gate_prediction(variant_name, raw_base, raw_repair, gate_cfg, baseline_detail=None):
    raw_gate = apply_tournament_gated_outcome(
        raw_base,
        raw_repair,
        gate_cfg['gate_map'],
        default_alpha=gate_cfg['default_alpha'],
    )
    tail_prob = raw_base[['match_id'] + TAIL_COLS].copy()
    decoder_grid, best_decoder, pred = tune_tail_aware_decoder_from_raw_outputs(
        raw_gate,
        tail_prob,
        TAIL_GRID,
        scoreline_prior_lookup,
        top_n=3,
    )
    pred_match = raw_gate.merge(pred, on='match_id', how='left', validate='one_to_one')
    detail = prediction_detail_table(pred_match, variant_name)
    summary = summarize_prediction_detail(detail, variant_name)
    summary['mean_gate_alpha'] = float(raw_gate['gate_alpha'].mean())
    for value, label in [(0.0, 'n_alpha_0'), (0.25, 'n_alpha_025'), (0.5, 'n_alpha_050'), (0.75, 'n_alpha_075'), (1.0, 'n_alpha_100')]:
        summary[label] = int(np.isclose(raw_gate['gate_alpha'].to_numpy(float), value).sum())

    if baseline_detail is not None:
        compare = detail[['match_id', 'pred_team_a_goals', 'pred_team_b_goals', 'official_loss', 'tournament_weight']].merge(
            baseline_detail[['match_id', 'pred_team_a_goals', 'pred_team_b_goals', 'official_loss']],
            on='match_id',
            how='left',
            suffixes=('', '_v0'),
            validate='one_to_one',
        )
        changed = (
            (compare['pred_team_a_goals'] != compare['pred_team_a_goals_v0'])
            | (compare['pred_team_b_goals'] != compare['pred_team_b_goals_v0'])
        )
        delta = compare['official_loss'] - compare['official_loss_v0']
        summary['n_changed_vs_v0'] = int(changed.sum())
        summary['changed_rate_vs_v0'] = float(changed.mean())
        summary['n_improved_changed'] = int(((delta < 0) & changed).sum())
        summary['n_worsened_changed'] = int(((delta > 0) & changed).sum())
        summary['n_neutral_changed'] = int(((delta == 0) & changed).sum())
        summary['weighted_loss_delta_vs_v0'] = float((delta * compare['tournament_weight']).sum() / compare['tournament_weight'].sum())
    else:
        summary['n_changed_vs_v0'] = 0
        summary['changed_rate_vs_v0'] = 0.0
        summary['n_improved_changed'] = 0
        summary['n_worsened_changed'] = 0
        summary['n_neutral_changed'] = 0
        summary['weighted_loss_delta_vs_v0'] = 0.0

    return raw_gate, pred_match, detail, summary, decoder_grid, best_decoder


log_section('EXP09D gate selection baseline for EXP11B')
GATE_RESULTS = {}
gate_metric_rows = []
v0_detail = None

for variant_name, cfg in GATE_CONFIGS.items():
    raw_gate, pred_match, detail, summary, decoder_grid, best_decoder = evaluate_gate_prediction(
        variant_name,
        raw_base_valid,
        raw_repair_valid,
        cfg,
        baseline_detail=v0_detail,
    )
    if variant_name == 'V0_baseline_outcome':
        v0_detail = detail.copy()
        raw_gate, pred_match, detail, summary, decoder_grid, best_decoder = evaluate_gate_prediction(
            variant_name,
            raw_base_valid,
            raw_repair_valid,
            cfg,
            baseline_detail=v0_detail,
        )

    pred_path = PRED_DIR / f'exp09d_valid_pred_match_{variant_name}.csv'
    decoder_grid_path = SUM_DIR / f'exp09d_decoder_grid_{variant_name}.csv'
    pred_match.to_csv(pred_path, index=False)
    decoder_grid.to_csv(decoder_grid_path, index=False)
    log_saved(pred_path)
    log_saved(decoder_grid_path)

    GATE_RESULTS[variant_name] = {
        'raw': raw_gate,
        'pred_match': pred_match,
        'detail': detail,
        'summary': summary,
        'config': cfg,
        'decoder_grid': decoder_grid,
        'best_decoder': best_decoder,
    }
    gate_metric_rows.append(summary)

gate_variant_metrics_df = pd.DataFrame(gate_metric_rows)
v0_gate = gate_variant_metrics_df[gate_variant_metrics_df['variant_name'].eq('V0_baseline_outcome')].iloc[0]
v1_gate = gate_variant_metrics_df[gate_variant_metrics_df['variant_name'].eq('V1_full_repair')].iloc[0]
gate_variant_metrics_df['awmae_gain_vs_v0'] = float(v0_gate['awmae']) - gate_variant_metrics_df['awmae']
gate_variant_metrics_df = gate_variant_metrics_df.sort_values('awmae').reset_index(drop=True)
gate_variant_metrics_df.to_csv(SUM_DIR / 'exp09d_gate_variant_metrics.csv', index=False)
display(gate_variant_metrics_df)
log_saved(SUM_DIR / 'exp09d_gate_variant_metrics.csv')

# Safety-first EXP09D gate selection. This mirrors the tournament-gated repair spirit.
def select_exp09d_gate(metrics_df):
    metrics = metrics_df.copy()
    v0 = metrics[metrics['variant_name'].eq('V0_baseline_outcome')].iloc[0]
    v1 = metrics[metrics['variant_name'].eq('V1_full_repair')].iloc[0]
    gate_candidates = metrics[metrics['variant_name'].isin(['V2_hard_helpful_gate', 'V3_conservative_soft_gate'])].copy()
    safe_rows = []
    for _, row in gate_candidates.iterrows():
        safe = (
            (float(row['awmae']) < float(v0['awmae']))
            and (float(row['exact_rate']) >= float(v0['exact_rate']) - 0.010)
            and (float(row['gd_rate']) >= float(v0['gd_rate']) - 0.015)
            and (float(row['top3_scoreline_share']) <= float(v0['top3_scoreline_share']) + 0.030)
            and (1.0 <= float(row['mean_pred_total']) <= 4.5)
        )
        if safe:
            safe_rows.append(row)
    if safe_rows:
        best = pd.DataFrame(safe_rows).sort_values('awmae').iloc[0]
        return str(best['variant_name']), 'A', 'Tournament gate improves AW-MAE and passes EXP09D safety checks.'

    v1_safe = (
        (float(v1['awmae']) < float(v0['awmae']))
        and (float(v1['exact_rate']) >= float(v0['exact_rate']) - 0.010)
        and (float(v1['gd_rate']) >= float(v0['gd_rate']) - 0.015)
        and (float(v1['top3_scoreline_share']) <= float(v0['top3_scoreline_share']) + 0.030)
        and (1.0 <= float(v1['mean_pred_total']) <= 4.5)
    )
    if v1_safe:
        return 'V1_full_repair', 'C', 'Full repair remains best and passes safety; using EXP09D full repair as baseline.'
    return 'V0_baseline_outcome', 'D', 'No repair gate passed safety; fallback to baseline outcome.'

selected_gate_name, exp09d_gate_decision_code, exp09d_gate_decision_text = select_exp09d_gate(gate_variant_metrics_df)
selected_gate_cfg = GATE_CONFIGS[selected_gate_name]
selected_exp09d_decoder = GATE_RESULTS[selected_gate_name]['best_decoder']
raw_gate_valid = GATE_RESULTS[selected_gate_name]['raw'].copy()
exp09d_detail = GATE_RESULTS[selected_gate_name]['detail'].copy()

selected_gate_df = pd.DataFrame([{
    'selected_gate': selected_gate_name,
    'decision_code': exp09d_gate_decision_code,
    'decision_text': exp09d_gate_decision_text,
    'selected_gate_awmae': float(gate_variant_metrics_df[gate_variant_metrics_df['variant_name'].eq(selected_gate_name)]['awmae'].iloc[0]),
    'selected_decoder': str(selected_exp09d_decoder),
}])
display(selected_gate_df)
selected_gate_df.to_csv(SUM_DIR / 'exp09d_selected_gate_for_exp11a.csv', index=False)
log_saved(SUM_DIR / 'exp09d_selected_gate_for_exp11a.csv')


 %% [markdown]

 # 14. Ordinal Goal Target Builder



 Bagian ini membuat target ordinal `goal >= k` untuk team_a dan team_b. Target ini dipakai untuk

 melatih binary classifiers yang mempelajari distribusi peluang gol, bukan hanya satu angka prediksi.

 Output yang diharapkan adalah ringkasan positive rate tiap threshold.

In [ ]:
MAX_ORDINAL_GOALS = 10
MAX_DECODE_GOALS = 9
ORDINAL_N_ESTIMATORS = 650


def build_ordinal_goal_targets(y_goal_a, y_goal_b, max_goal=10):
    out = pd.DataFrame()
    y_a = pd.Series(y_goal_a).astype(int)
    y_b = pd.Series(y_goal_b).astype(int)
    for k in range(1, max_goal + 1):
        out[f'a_goal_ge_{k}'] = (y_a >= k).astype(int)
        out[f'b_goal_ge_{k}'] = (y_b >= k).astype(int)
    return out

ordinal_train_targets = build_ordinal_goal_targets(train_sup_full['y_goal_a'], train_sup_full['y_goal_b'], MAX_ORDINAL_GOALS)
ordinal_valid_targets = build_ordinal_goal_targets(valid_sup_full['y_goal_a'], valid_sup_full['y_goal_b'], MAX_ORDINAL_GOALS)
ordinal_target_summary_rows = []
for side in ['a', 'b']:
    for k in range(1, MAX_ORDINAL_GOALS + 1):
        col = f'{side}_goal_ge_{k}'
        ordinal_target_summary_rows.append({
            'side': side,
            'threshold': k,
            'train_positive_rate': float(ordinal_train_targets[col].mean()),
            'valid_positive_rate': float(ordinal_valid_targets[col].mean()),
            'train_positive_count': int(ordinal_train_targets[col].sum()),
            'valid_positive_count': int(ordinal_valid_targets[col].sum()),
        })
ordinal_target_summary_df = pd.DataFrame(ordinal_target_summary_rows)
ordinal_target_summary_df.to_csv(SUM_DIR / 'ordinal_target_summary.csv', index=False)
display(ordinal_target_summary_df)
log_saved(SUM_DIR / 'ordinal_target_summary.csv')


 # 15. Train Ordinal Goal Distribution Models



 Bagian ini melatih ordinal classifier per gender, per side, dan per threshold. Feature set default

 memakai EXP05A agar tidak langsung mengulang noise EXP09A all-head. Booster mengikuti pilihan EXP09D

 per gender supaya pipeline tetap preserve.

In [ ]:
class ConstantBinaryModel:
    def __init__(self, prob=0.0):
        self.prob = float(np.clip(prob, 1e-6, 1 - 1e-6))
        self.classes_ = np.array([0, 1])
    def predict_proba(self, X):
        n = len(X)
        return np.column_stack([np.full(n, 1.0 - self.prob), np.full(n, self.prob)])


def fit_binary_model_safe(Xtr, ytr, Xva, yva, booster, params, cats):
    ytr = pd.Series(ytr).astype(int)
    if ytr.nunique() < 2:
        return ConstantBinaryModel(float(ytr.mean()))
    return fit_tail_model(Xtr, ytr, Xva, pd.Series(yva).astype(int), booster, params, cats)


def train_ordinal_models(train_df, valid_df, final_selected, feature_cols, cat_features, max_goal=10, n_estimators=650):
    models = {}
    config_rows = []
    threshold_metric_rows = []
    for gender, cfg in final_selected.items():
        booster, regime = cfg['booster'], cfg['regime']
        tr_all = filter_regime_train(train_df, regime)
        tr = tr_all[tr_all['gender'].astype(str).eq(str(gender))].copy()
        va = valid_df[valid_df['gender'].astype(str).eq(str(gender))].copy()
        if len(tr) == 0 or len(va) == 0:
            log_warn(f'Skip ordinal gender={gender} because train/valid is empty')
            continue
        cols = ['match_id', 'gender', 'date', 'y_goal_a', 'y_goal_b'] + list(feature_cols)
        tr = select_unique_columns(tr, cols)
        va = select_unique_columns(va, cols)
        Xtr = prepare_model_frame(tr, feature_cols, cat_features)
        Xva = prepare_model_frame(va, feature_cols, cat_features)
        params = make_default_params(booster, 'tail', SEED, n_estimators)
        models[str(gender)] = {
            'booster': booster,
            'regime': regime,
            'feature_cols': list(feature_cols),
            'cat_features': list(cat_features),
            'side_models': {'a': {}, 'b': {}},
        }
        log_info(f'Train ordinal models | gender={gender} | booster={booster} | regime={regime} | n_train={len(tr)} | n_valid={len(va)}')
        for side, target_col in [('a', 'y_goal_a'), ('b', 'y_goal_b')]:
            ytr_goal = pd.to_numeric(tr[target_col], errors='coerce').fillna(0).astype(int)
            yva_goal = pd.to_numeric(va[target_col], errors='coerce').fillna(0).astype(int)
            for k in range(1, max_goal + 1):
                ytr_bin = (ytr_goal >= k).astype(int)
                yva_bin = (yva_goal >= k).astype(int)
                model = fit_binary_model_safe(Xtr, ytr_bin, Xva, yva_bin, booster, params, cat_features)
                models[str(gender)]['side_models'][side][k] = model
                pred = predict_pos(model, Xva)
                threshold_metric_rows.append({
                    'gender': str(gender),
                    'side': side,
                    'threshold': k,
                    'booster': booster,
                    'regime': regime,
                    'valid_positive_rate': float(yva_bin.mean()),
                    'pred_positive_mean': float(np.mean(pred)),
                    'brier': float(np.mean((pred - yva_bin.to_numpy(float)) ** 2)),
                })
        config_rows.append({
            'gender': str(gender),
            'booster': booster,
            'regime': regime,
            'n_features': len(feature_cols),
            'n_cat_features': len(cat_features),
            'max_ordinal_goals': max_goal,
            'n_estimators': n_estimators,
        })
    return models, pd.DataFrame(config_rows), pd.DataFrame(threshold_metric_rows)

log_section('Train ordinal goal distribution models')
ordinal_models, ordinal_model_config_df, ordinal_threshold_metrics_df = train_ordinal_models(
    train_sup_full,
    valid_sup_full,
    final_selected,
    exp05a_feature_cols,
    exp05a_cat_features,
    max_goal=MAX_ORDINAL_GOALS,
    n_estimators=ORDINAL_N_ESTIMATORS,
)
ordinal_model_config_df.to_csv(SUM_DIR / 'ordinal_model_config.csv', index=False)
ordinal_threshold_metrics_df.to_csv(SUM_DIR / 'ordinal_threshold_metrics.csv', index=False)
display(ordinal_model_config_df)
display(ordinal_threshold_metrics_df.head(40))
log_saved(SUM_DIR / 'ordinal_model_config.csv')
log_saved(SUM_DIR / 'ordinal_threshold_metrics.csv')


 # 16. Build Goal PMF from Ordinal Probabilities



 Bagian ini mengubah probability `P(goal >= k)` menjadi PMF `P(goal = k)`. Probability dibersihkan

 agar monotonic, non-negative, dan sum-to-one. Poisson PMF dari continuous goal EXP09D juga dibuat

 untuk variant blend.

In [ ]:
def enforce_monotonic_ge_probs(p_ge):
    p = np.asarray(p_ge, dtype=float)
    p = np.clip(p, 1e-6, 1 - 1e-6)
    for k in range(1, p.shape[1]):
        p[:, k] = np.minimum(p[:, k], p[:, k - 1])
    return p


def ge_probs_to_pmf(p_ge):
    p_ge = enforce_monotonic_ge_probs(p_ge)
    n, kmax = p_ge.shape
    pmf = np.zeros((n, kmax + 1), dtype=float)
    pmf[:, 0] = 1.0 - p_ge[:, 0]
    for k in range(1, kmax):
        pmf[:, k] = p_ge[:, k - 1] - p_ge[:, k]
    pmf[:, kmax] = p_ge[:, kmax - 1]
    pmf = np.clip(pmf, 1e-12, 1.0)
    pmf = pmf / pmf.sum(axis=1, keepdims=True)
    return pmf


def poisson_pmf_from_lambda(lam, max_goal=10):
    lam = np.asarray(lam, dtype=float)
    lam = np.clip(lam, 1e-6, 20)
    pmf = np.zeros((len(lam), max_goal + 1), dtype=float)
    pmf[:, 0] = np.exp(-lam)
    for k in range(1, max_goal + 1):
        pmf[:, k] = pmf[:, k - 1] * lam / k
    pmf = np.clip(pmf, 1e-12, 1.0)
    pmf = pmf / pmf.sum(axis=1, keepdims=True)
    return pmf


def blend_pmfs(pmf_ordinal, pmf_poisson, blend_ordinal=0.75):
    p = float(blend_ordinal) * pmf_ordinal + (1 - float(blend_ordinal)) * pmf_poisson
    p = np.clip(p, 1e-12, 1.0)
    p = p / p.sum(axis=1, keepdims=True)
    return p


def predict_ordinal_ge_probs(df, ordinal_models, side='a', max_goal=10):
    p = np.full((len(df), max_goal), 1e-6, dtype=float)
    df_work = df.reset_index(drop=False).rename(columns={'index': '_orig_index'})
    for gender, bundle in ordinal_models.items():
        mask = df_work['gender'].astype(str).eq(str(gender))
        if not mask.any():
            continue
        sub = df_work.loc[mask].copy()
        X = prepare_model_frame(sub, bundle['feature_cols'], bundle['cat_features'])
        for k in range(1, max_goal + 1):
            model = bundle['side_models'][side][k]
            p[sub['_orig_index'].to_numpy(int), k - 1] = predict_pos(model, X)
    return enforce_monotonic_ge_probs(p)

log_section('Build ordinal and Poisson PMFs for validation')
# PMF rows must follow raw_gate_valid order because MBR consumes raw_gate_valid row-by-row.
valid_feature_for_pmf = raw_gate_valid[['match_id']].merge(valid_decode_df, on='match_id', how='left', validate='one_to_one')
suffix_cols = [c for c in valid_feature_for_pmf.columns if c.endswith('_x') or c.endswith('_y')]
assert len(suffix_cols) == 0, f'valid_feature_for_pmf has bad suffix columns: {suffix_cols}'
valid_ge_a = predict_ordinal_ge_probs(valid_feature_for_pmf, ordinal_models, side='a', max_goal=MAX_ORDINAL_GOALS)
valid_ge_b = predict_ordinal_ge_probs(valid_feature_for_pmf, ordinal_models, side='b', max_goal=MAX_ORDINAL_GOALS)
valid_pmf_ord_a = ge_probs_to_pmf(valid_ge_a)
valid_pmf_ord_b = ge_probs_to_pmf(valid_ge_b)
valid_pmf_pois_a = poisson_pmf_from_lambda(raw_gate_valid['pred_goal_a_cont'], max_goal=MAX_ORDINAL_GOALS)
valid_pmf_pois_b = poisson_pmf_from_lambda(raw_gate_valid['pred_goal_b_cont'], max_goal=MAX_ORDINAL_GOALS)

pmf_quality_summary_df = pd.DataFrame([
    {'pmf_type': 'ordinal', 'side': 'a', 'mean_goal_from_pmf': float((valid_pmf_ord_a * np.arange(MAX_ORDINAL_GOALS + 1)).sum(axis=1).mean()), 'actual_mean_goal': float(valid_sup_full['y_goal_a'].mean()), 'mean_sum': float(valid_pmf_ord_a.sum(axis=1).mean())},
    {'pmf_type': 'ordinal', 'side': 'b', 'mean_goal_from_pmf': float((valid_pmf_ord_b * np.arange(MAX_ORDINAL_GOALS + 1)).sum(axis=1).mean()), 'actual_mean_goal': float(valid_sup_full['y_goal_b'].mean()), 'mean_sum': float(valid_pmf_ord_b.sum(axis=1).mean())},
    {'pmf_type': 'poisson_from_cont', 'side': 'a', 'mean_goal_from_pmf': float((valid_pmf_pois_a * np.arange(MAX_ORDINAL_GOALS + 1)).sum(axis=1).mean()), 'actual_mean_goal': float(valid_sup_full['y_goal_a'].mean()), 'mean_sum': float(valid_pmf_pois_a.sum(axis=1).mean())},
    {'pmf_type': 'poisson_from_cont', 'side': 'b', 'mean_goal_from_pmf': float((valid_pmf_pois_b * np.arange(MAX_ORDINAL_GOALS + 1)).sum(axis=1).mean()), 'actual_mean_goal': float(valid_sup_full['y_goal_b'].mean()), 'mean_sum': float(valid_pmf_pois_b.sum(axis=1).mean())},
])
pmf_quality_summary_df.to_csv(SUM_DIR / 'pmf_quality_summary.csv', index=False)
display(pmf_quality_summary_df)
log_saved(SUM_DIR / 'pmf_quality_summary.csv')



 # 17. MBR Penalty Function



 Bagian ini membuat decoder MBR dengan penalty tambahan. Basisnya tetap EXP11A: M match memakai

 ordinal PMF MBR dan W match fallback ke EXP09D original. Penalty yang diuji hanya memodifikasi

 risk candidate scoreline, bukan mengubah training model, fitur, atau submission mapping.

In [ ]:
def build_loss_matrix(max_pmf_goal=10, max_decode_goals=9):
    true_scores = [(a, b) for a in range(max_pmf_goal + 1) for b in range(max_pmf_goal + 1)]
    candidates = [(a, b) for a in range(max_decode_goals + 1) for b in range(max_decode_goals + 1)]
    loss = np.zeros((len(true_scores), len(candidates)), dtype=float)
    for i, (ta, tb) in enumerate(true_scores):
        for j, (pa, pb) in enumerate(candidates):
            loss[i, j] = official_match_loss(ta, tb, pa, pb)
    return true_scores, candidates, loss

TRUE_SCORES_MBR, CANDIDATES_MBR, LOSS_MATRIX_MBR = build_loss_matrix(MAX_ORDINAL_GOALS, MAX_DECODE_GOALS)


def mbr_decode_from_pmfs_with_penalty(
    raw_df,
    pmf_a,
    pmf_b,
    max_decode_goals=9,
    outcome_weight=0.0,
    gd_anchor_weight=0.0,
    total_anchor_weight=0.0,
    under_bias_weight=0.0,
):
    """
    Decode scoreline with expected AW-MAE risk + lightweight penalties.
    Uses existing _outcome mapping from EXP05A/EXP09D, so outcome probability columns stay aligned.
    """
    candidates = [(a, b) for a in range(max_decode_goals + 1) for b in range(max_decode_goals + 1)]
    if max_decode_goals == MAX_DECODE_GOALS and pmf_a.shape[1] == MAX_ORDINAL_GOALS + 1:
        loss_matrix = LOSS_MATRIX_MBR
    else:
        _, _, loss_matrix = build_loss_matrix(pmf_a.shape[1] - 1, max_decode_goals)

    joint_flat = (pmf_a[:, :, None] * pmf_b[:, None, :]).reshape(len(raw_df), -1)
    joint_flat = joint_flat / np.maximum(joint_flat.sum(axis=1, keepdims=True), 1e-12)
    base_risks = joint_flat @ loss_matrix
    risks = base_risks.copy()

    cand_a = np.array([c[0] for c in candidates], dtype=float)
    cand_b = np.array([c[1] for c in candidates], dtype=float)
    cand_total = cand_a + cand_b
    cand_gd = cand_a - cand_b

    if outcome_weight > 0:
        p_out = raw_df[['pred_outcome_proba_0', 'pred_outcome_proba_1', 'pred_outcome_proba_2']].to_numpy(float)
        p_out = np.clip(p_out, 1e-9, 1.0)
        cand_out = np.array([_outcome(a, b) for a, b in candidates], dtype=int)
        risks += float(outcome_weight) * (-np.log(p_out[:, cand_out] + 1e-9))

    cont_a = pd.to_numeric(raw_df['pred_goal_a_cont'], errors='coerce').fillna(1.0).to_numpy(float)
    cont_b = pd.to_numeric(raw_df['pred_goal_b_cont'], errors='coerce').fillna(1.0).to_numpy(float)
    cont_total = cont_a + cont_b
    cont_gd = cont_a - cont_b

    if gd_anchor_weight > 0:
        risks += float(gd_anchor_weight) * np.abs(cand_gd[None, :] - cont_gd[:, None])
    if total_anchor_weight > 0:
        risks += float(total_anchor_weight) * np.abs(cand_total[None, :] - cont_total[:, None])
    if under_bias_weight > 0:
        risks += float(under_bias_weight) * np.maximum(0.0, cont_total[:, None] - cand_total[None, :])

    idx = np.argmin(risks, axis=1)
    return pd.DataFrame({
        'match_id': raw_df['match_id'].values,
        'pred_team_a_goals': cand_a[idx].astype(int),
        'pred_team_b_goals': cand_b[idx].astype(int),
        'mbr_risk': risks[np.arange(len(raw_df)), idx].astype(float),
        'mbr_base_risk': base_risks[np.arange(len(raw_df)), idx].astype(float),
    })


def decode_exp11b_policy(raw_gate_df, pmf_a, pmf_b, exp09d_pred_df, cfg):
    """EXP11B selected policy: M uses ordinal MBR with penalty, W keeps EXP09D original."""
    pred_ord = mbr_decode_from_pmfs_with_penalty(
        raw_gate_df,
        pmf_a,
        pmf_b,
        max_decode_goals=MAX_DECODE_GOALS,
        outcome_weight=float(cfg.get('outcome_weight', 0.0)),
        gd_anchor_weight=float(cfg.get('gd_anchor_weight', 0.0)),
        total_anchor_weight=float(cfg.get('total_anchor_weight', 0.0)),
        under_bias_weight=float(cfg.get('under_bias_weight', 0.0)),
    )
    combo = raw_gate_df[['match_id', 'gender']].merge(pred_ord, on='match_id', how='left', validate='one_to_one')
    combo = combo.merge(
        exp09d_pred_df[['match_id', 'pred_team_a_goals', 'pred_team_b_goals']],
        on='match_id', how='left', suffixes=('', '_exp09d'), validate='one_to_one'
    )
    is_m = combo['gender'].astype(str).eq('M')
    combo['pred_team_a_goals'] = np.where(is_m, combo['pred_team_a_goals'], combo['pred_team_a_goals_exp09d']).astype(int)
    combo['pred_team_b_goals'] = np.where(is_m, combo['pred_team_b_goals'], combo['pred_team_b_goals_exp09d']).astype(int)
    return combo[['match_id', 'pred_team_a_goals', 'pred_team_b_goals']]


 # 18. Penalty Config Grid



 Bagian ini mendefinisikan grid kecil penalty config. B0 adalah EXP11A original tanpa penalty.

 Variant lain menambahkan outcome penalty, GD anchor penalty, total anchor penalty, kombinasi kecil,

 dan underprediction correction secara terkendali agar runtime dan risiko overfit tetap aman.

In [ ]:
MBR_PENALTY_CONFIGS = [
    {'variant_name': 'B0_exp11a_original', 'outcome_weight': 0.00, 'gd_anchor_weight': 0.00, 'total_anchor_weight': 0.00, 'under_bias_weight': 0.00},
    {'variant_name': 'B1_outcome_003', 'outcome_weight': 0.03, 'gd_anchor_weight': 0.00, 'total_anchor_weight': 0.00, 'under_bias_weight': 0.00},
    {'variant_name': 'B1_outcome_005', 'outcome_weight': 0.05, 'gd_anchor_weight': 0.00, 'total_anchor_weight': 0.00, 'under_bias_weight': 0.00},
    {'variant_name': 'B1_outcome_008', 'outcome_weight': 0.08, 'gd_anchor_weight': 0.00, 'total_anchor_weight': 0.00, 'under_bias_weight': 0.00},
    {'variant_name': 'B2_gd_002', 'outcome_weight': 0.00, 'gd_anchor_weight': 0.02, 'total_anchor_weight': 0.00, 'under_bias_weight': 0.00},
    {'variant_name': 'B2_gd_005', 'outcome_weight': 0.00, 'gd_anchor_weight': 0.05, 'total_anchor_weight': 0.00, 'under_bias_weight': 0.00},
    {'variant_name': 'B3_total_002', 'outcome_weight': 0.00, 'gd_anchor_weight': 0.00, 'total_anchor_weight': 0.02, 'under_bias_weight': 0.00},
    {'variant_name': 'B3_total_005', 'outcome_weight': 0.00, 'gd_anchor_weight': 0.00, 'total_anchor_weight': 0.05, 'under_bias_weight': 0.00},
    {'variant_name': 'B4_combo_o003_g002_t002', 'outcome_weight': 0.03, 'gd_anchor_weight': 0.02, 'total_anchor_weight': 0.02, 'under_bias_weight': 0.00},
    {'variant_name': 'B4_combo_o005_g002_t002', 'outcome_weight': 0.05, 'gd_anchor_weight': 0.02, 'total_anchor_weight': 0.02, 'under_bias_weight': 0.00},
    {'variant_name': 'B4_combo_o003_g005_t002', 'outcome_weight': 0.03, 'gd_anchor_weight': 0.05, 'total_anchor_weight': 0.02, 'under_bias_weight': 0.00},
    {'variant_name': 'B5_under_002', 'outcome_weight': 0.00, 'gd_anchor_weight': 0.00, 'total_anchor_weight': 0.00, 'under_bias_weight': 0.02},
    {'variant_name': 'B5_under_005', 'outcome_weight': 0.00, 'gd_anchor_weight': 0.00, 'total_anchor_weight': 0.00, 'under_bias_weight': 0.05},
]

mbr_penalty_config_df = pd.DataFrame(MBR_PENALTY_CONFIGS)
mbr_penalty_config_df.to_csv(SUM_DIR / 'mbr_penalty_config.csv', index=False)
display(mbr_penalty_config_df)
log_saved(SUM_DIR / 'mbr_penalty_config.csv')


 # 19. Evaluate Penalty Variants



 Bagian ini mengevaluasi setiap penalty variant pada validation. Semua variant dibandingkan terhadap

 B0 EXP11A original. Metric utama meliputi AW-MAE, base MAE, exact, outcome, GD, bias, scoreline

 concentration, dan perubahan prediksi pada match yang terdampak penalty.

In [ ]:
def add_changed_metrics_vs_baseline(summary, detail, baseline_detail, suffix='b0'):
    compare = detail[['match_id', 'pred_team_a_goals', 'pred_team_b_goals', 'official_loss', 'tournament_weight']].merge(
        baseline_detail[['match_id', 'pred_team_a_goals', 'pred_team_b_goals', 'official_loss']],
        on='match_id', how='left', suffixes=('', f'_{suffix}'), validate='one_to_one'
    )
    changed = (
        (compare['pred_team_a_goals'] != compare[f'pred_team_a_goals_{suffix}'])
        | (compare['pred_team_b_goals'] != compare[f'pred_team_b_goals_{suffix}'])
    )
    delta = compare['official_loss'] - compare[f'official_loss_{suffix}']
    summary[f'n_changed_vs_{suffix}'] = int(changed.sum())
    summary[f'changed_rate_vs_{suffix}'] = float(changed.mean())
    summary['n_improved_changed'] = int(((delta < 0) & changed).sum())
    summary['n_worsened_changed'] = int(((delta > 0) & changed).sum())
    summary['n_neutral_changed'] = int(((delta == 0) & changed).sum())
    summary[f'weighted_loss_delta_vs_{suffix}'] = float((delta * compare['tournament_weight']).sum() / compare['tournament_weight'].sum())
    return summary


def evaluate_penalty_variant(cfg, baseline_detail=None):
    variant_name = cfg['variant_name']
    pred = decode_exp11b_policy(raw_gate_valid, valid_pmf_ord_a, valid_pmf_ord_b, GATE_RESULTS[selected_gate_name]['pred_match'], cfg)
    pred_match = raw_gate_valid.merge(pred, on='match_id', how='left', validate='one_to_one')
    detail = prediction_detail_table(pred_match, variant_name)
    summary = summarize_prediction_detail(detail, variant_name)
    summary['outcome_weight'] = float(cfg.get('outcome_weight', 0.0))
    summary['gd_anchor_weight'] = float(cfg.get('gd_anchor_weight', 0.0))
    summary['total_anchor_weight'] = float(cfg.get('total_anchor_weight', 0.0))
    summary['under_bias_weight'] = float(cfg.get('under_bias_weight', 0.0))
    if baseline_detail is None:
        summary['n_changed_vs_b0'] = 0
        summary['changed_rate_vs_b0'] = 0.0
        summary['n_improved_changed'] = 0
        summary['n_worsened_changed'] = 0
        summary['n_neutral_changed'] = 0
        summary['weighted_loss_delta_vs_b0'] = 0.0
    else:
        summary = add_changed_metrics_vs_baseline(summary, detail, baseline_detail, suffix='b0')
    return pred_match, detail, summary

log_section('Evaluate MBR penalty variants')
PENALTY_RESULTS = {}
penalty_rows = []
b0_detail = None
for cfg in MBR_PENALTY_CONFIGS:
    pred_match, detail, summary = evaluate_penalty_variant(cfg, baseline_detail=b0_detail)
    if cfg['variant_name'] == 'B0_exp11a_original':
        b0_detail = detail.copy()
        summary = add_changed_metrics_vs_baseline(summary, detail, b0_detail, suffix='b0')
    save_name = cfg['variant_name'].lower()
    pred_path = PRED_DIR / f'valid_pred_match_{save_name}.csv'
    pred_match.to_csv(pred_path, index=False)
    log_saved(pred_path)
    PENALTY_RESULTS[cfg['variant_name']] = {'config': cfg, 'pred_match': pred_match, 'detail': detail, 'summary': summary}
    penalty_rows.append(summary)

mbr_penalty_metrics_df = pd.DataFrame(penalty_rows).sort_values('awmae').reset_index(drop=True)
mbr_penalty_metrics_df.to_csv(SUM_DIR / 'mbr_penalty_metrics.csv', index=False)
display(mbr_penalty_metrics_df)
log_saved(SUM_DIR / 'mbr_penalty_metrics.csv')

changed_cols = ['variant_name', 'n_changed_vs_b0', 'changed_rate_vs_b0', 'n_improved_changed', 'n_worsened_changed', 'n_neutral_changed', 'weighted_loss_delta_vs_b0']
changed_prediction_df = mbr_penalty_metrics_df[[c for c in changed_cols if c in mbr_penalty_metrics_df.columns]].copy()
changed_prediction_df.to_csv(SUM_DIR / 'mbr_penalty_changed_prediction_analysis.csv', index=False)
display(changed_prediction_df)
log_saved(SUM_DIR / 'mbr_penalty_changed_prediction_analysis.csv')

scoreline_dist = scoreline_distribution(pd.concat([v['detail'] for v in PENALTY_RESULTS.values()], ignore_index=True))
scoreline_dist.to_csv(SUM_DIR / 'scoreline_distribution.csv', index=False)
display(scoreline_dist.groupby('variant_name').head(10).reset_index(drop=True))
log_saved(SUM_DIR / 'scoreline_distribution.csv')


 # 20. Domain Error Analysis



 Bagian ini mengecek performa penalty variant berdasarkan gender, tournament_structure, dan blowout /

 high-total flags. Fokusnya adalah memastikan penalty tidak memperbaiki satu metrik dengan cara

 merusak women match, tournament tertentu, atau kasus blowout.

In [ ]:
log_section('Domain error analysis for MBR penalty variants')
all_penalty_detail_df = pd.concat([v['detail'] for v in PENALTY_RESULTS.values()], ignore_index=True)

def compute_domain_error_per_variant(detail_df, group_col):
    if group_col not in detail_df.columns:
        return pd.DataFrame()
    rows = []
    for (variant, key), g in detail_df.groupby(['variant_name', group_col], dropna=False):
        rows.append({
            'variant_name': variant,
            group_col: key,
            'n': len(g),
            'awmae': awmae_score(g['actual_team_a_goals'], g['actual_team_b_goals'], g['pred_team_a_goals'], g['pred_team_b_goals'], g['tournament']),
            'base_mae': float(g['base_mae'].mean()),
            'exact_rate': float(g['exact_hit'].mean()),
            'outcome_rate': float(g['outcome_hit'].mean()),
            'gd_rate': float(g['gd_hit'].mean()),
            'total_goal_bias': float(((g['pred_team_a_goals'] + g['pred_team_b_goals']) - (g['actual_team_a_goals'] + g['actual_team_b_goals'])).mean()),
        })
    return pd.DataFrame(rows).sort_values(['variant_name', 'awmae']).reset_index(drop=True)

domain_gender_df = compute_domain_error_per_variant(all_penalty_detail_df, 'gender')
domain_tournament_df = compute_domain_error_per_variant(all_penalty_detail_df, 'tournament_structure')
domain_blowout_df = pd.concat([
    compute_domain_error_per_variant(all_penalty_detail_df, 'is_blowout_5plus').assign(domain='is_blowout_5plus'),
    compute_domain_error_per_variant(all_penalty_detail_df, 'is_blowout_7plus').assign(domain='is_blowout_7plus'),
    compute_domain_error_per_variant(all_penalty_detail_df, 'is_high_total_6plus').assign(domain='is_high_total_6plus'),
], ignore_index=True)

domain_gender_df.to_csv(SUM_DIR / 'domain_error_gender.csv', index=False)
domain_tournament_df.to_csv(SUM_DIR / 'domain_error_tournament_structure.csv', index=False)
domain_blowout_df.to_csv(SUM_DIR / 'domain_error_blowout.csv', index=False)
display(domain_gender_df.head(40))
display(domain_tournament_df.head(50))
log_saved(SUM_DIR / 'domain_error_gender.csv')
log_saved(SUM_DIR / 'domain_error_tournament_structure.csv')
log_saved(SUM_DIR / 'domain_error_blowout.csv')


 # 21. Final Decision



 Bagian ini memilih penalty terbaik berdasarkan AW-MAE dan safety check. Jika penalty terbaik tidak

 aman, final fallback ke B0 EXP11A original. Final decision disimpan sebagai JSON dan ditampilkan

 sebagai DataFrame agar output notebook tetap rapi.

In [ ]:
log_section('Final decision for EXP11B')
metrics = mbr_penalty_metrics_df.copy()
b0 = metrics[metrics['variant_name'].eq('B0_exp11a_original')].iloc[0]
metrics['awmae_gain_vs_b0'] = float(b0['awmae']) - metrics['awmae']
metrics['bias_delta_vs_b0'] = metrics['total_goal_bias'].astype(float) - float(b0['total_goal_bias'])

safe_candidates = []
for _, row in metrics[~metrics['variant_name'].eq('B0_exp11a_original')].iterrows():
    safe = (
        (float(row['awmae']) < float(b0['awmae']))
        and (float(row['base_mae']) <= float(b0['base_mae']) + 0.002)
        and (float(row['exact_rate']) >= float(b0['exact_rate']) - 0.006)
        and (float(row['outcome_rate']) >= float(b0['outcome_rate']) - 0.004)
        and (float(row['gd_rate']) >= float(b0['gd_rate']) - 0.006)
        and (float(row['top3_scoreline_share']) <= float(b0['top3_scoreline_share']) + 0.030)
        and (1.0 <= float(row['mean_pred_total']) <= 4.5)
        and (float(row['total_goal_bias']) >= float(b0['total_goal_bias']) - 0.050)
    )
    if safe:
        safe_candidates.append(row)

best_row = metrics.sort_values('awmae').iloc[0]
if safe_candidates:
    selected_row = pd.DataFrame(safe_candidates).sort_values('awmae').iloc[0]
    selected_penalty_name = str(selected_row['variant_name'])
    decision_code = 'A'
    decision_text = 'MBR penalty improves AW-MAE and passes safety checks.'
elif str(best_row['variant_name']) == 'B0_exp11a_original':
    selected_row = b0
    selected_penalty_name = 'B0_exp11a_original'
    decision_code = 'C'
    decision_text = 'EXP11A original remains best.'
else:
    selected_row = b0
    selected_penalty_name = 'B0_exp11a_original'
    if (
        float(best_row.get('outcome_rate', 0)) > float(b0.get('outcome_rate', 0))
        or float(best_row.get('gd_rate', 0)) > float(b0.get('gd_rate', 0))
        or float(best_row.get('total_goal_bias', -999)) > float(b0.get('total_goal_bias', -999))
    ):
        decision_code = 'B'
        decision_text = 'Penalty improves bias/outcome/GD signal but not enough to beat B0 safely, fallback to EXP11A.'
    else:
        decision_code = 'D'
        decision_text = 'Penalty variant is not safe by final rule, fallback to EXP11A original.'

selected_penalty_config = copy.deepcopy(PENALTY_RESULTS[selected_penalty_name]['config'])
selected_json = {
    'experiment': 'EXP11B',
    'base_pipeline': 'EXP05A-LITE-FIX-V2 / EXP09D / EXP11A',
    'selected_gate': selected_gate_name,
    'selected_policy': 'M ordinal MBR with W fallback to EXP09D',
    'selected_penalty_variant': selected_penalty_name,
    'selected_penalty_config': selected_penalty_config,
    'decision_code': decision_code,
    'decision_text': decision_text,
    'b0_awmae': float(b0['awmae']),
    'selected_awmae': float(selected_row['awmae']),
    'selected_awmae_gain_vs_b0': float(float(b0['awmae']) - float(selected_row['awmae'])),
}
with open(SUM_DIR / 'selected_mbr_penalty_config.json', 'w', encoding='utf-8') as f:
    json.dump(selected_json, f, indent=2, ensure_ascii=False)

final_decision = {
    'experiment': 'EXP11B',
    'main_change': 'Penalty tuning for EXP11A ordinal MBR risk function',
    'selected_gate': selected_gate_name,
    'selected_penalty_variant': selected_penalty_name,
    'decision_code': decision_code,
    'decision_text': decision_text,
    'b0_awmae': float(b0['awmae']),
    'selected_awmae': float(selected_row['awmae']),
    'selected_awmae_gain_vs_b0': float(float(b0['awmae']) - float(selected_row['awmae'])),
    'caveat': 'Training, feature engineering, PMF construction, and tournament gate are preserved. Only MBR risk penalty changes.',
}
with open(SUM_DIR / 'final_decision.json', 'w', encoding='utf-8') as f:
    json.dump(final_decision, f, indent=2, ensure_ascii=False)

final_decision_df = pd.DataFrame([{'key': k, 'value': str(v)} for k, v in final_decision.items()])
selected_mbr_penalty_config_df = pd.DataFrame([{'key': k, 'value': str(v)} for k, v in selected_json.items()])
final_decision_df.to_csv(SUM_DIR / 'final_decision_table.csv', index=False)
display(metrics.sort_values('awmae').reset_index(drop=True))
display(selected_mbr_penalty_config_df)
display(final_decision_df)
log_saved(SUM_DIR / 'selected_mbr_penalty_config.json')
log_saved(SUM_DIR / 'final_decision.json')
log_saved(SUM_DIR / 'final_decision_table.csv')


 # 22. Final Model Training dan Test Inference



 Bagian ini melatih ulang EXP09D base/repair dan ordinal models pada full train, membuat PMF test,

 lalu decode dengan selected penalty config. Jika B0 dipilih, output tetap sama secara konsep dengan

 EXP11A original: M ordinal MBR dan W EXP09D original.

In [ ]:
log_section('Final model training and test inference')
full_train_feature_df = train_feature_all.merge(y_full_train, on='match_id', how='left', validate='one_to_one')

full_base_models = train_final_gender_models_by_head_config(full_train_feature_df, valid_sup_full, final_selected, BASE_HEAD_CONFIG, 1200)
full_repair_models = train_final_gender_models_by_head_config(full_train_feature_df, valid_sup_full, final_selected, REPAIR_HEAD_CONFIG, 1200)

test_raw_base = predict_gender_split_raw_outputs_with_config(test_feature_full, full_base_models)
test_raw_repair = predict_gender_split_raw_outputs_with_config(test_feature_full, full_repair_models)
test_raw_gate = apply_tournament_gated_outcome(test_raw_base, test_raw_repair, selected_gate_cfg['gate_map'], default_alpha=selected_gate_cfg['default_alpha'])

test_pred_exp09d = decode_directional_tail_batch(
    test_raw_gate.drop(columns=TAIL_COLS, errors='ignore'),
    test_raw_base[['match_id'] + TAIL_COLS],
    selected_exp09d_decoder,
    scoreline_prior_lookup[int(selected_exp09d_decoder['MAX_GOALS'])],
)

full_ordinal_models, _, _ = train_ordinal_models(
    full_train_feature_df, valid_sup_full, final_selected, exp05a_feature_cols, exp05a_cat_features,
    max_goal=MAX_ORDINAL_GOALS, n_estimators=ORDINAL_N_ESTIMATORS,
)
test_feature_for_pmf = test_raw_gate[['match_id']].merge(test_feature_full, on='match_id', how='left', validate='one_to_one')
suffix_cols = [c for c in test_feature_for_pmf.columns if c.endswith('_x') or c.endswith('_y')]
assert len(suffix_cols) == 0, f'test_feature_for_pmf has bad suffix columns: {suffix_cols}'
test_ge_a = predict_ordinal_ge_probs(test_feature_for_pmf, full_ordinal_models, side='a', max_goal=MAX_ORDINAL_GOALS)
test_ge_b = predict_ordinal_ge_probs(test_feature_for_pmf, full_ordinal_models, side='b', max_goal=MAX_ORDINAL_GOALS)
test_pmf_ord_a = ge_probs_to_pmf(test_ge_a)
test_pmf_ord_b = ge_probs_to_pmf(test_ge_b)

test_pred = decode_exp11b_policy(test_raw_gate, test_pmf_ord_a, test_pmf_ord_b, test_pred_exp09d, selected_penalty_config)

test_pred_match_best = test_pred.merge(test_match_base[['match_id', 'team_a', 'team_b']], on='match_id', how='left', validate='one_to_one')
test_pred_match_best['selected_gate'] = selected_gate_name
test_pred_match_best['selected_penalty_variant'] = selected_penalty_name
test_pred_match_best.to_csv(PRED_DIR / 'test_pred_exp11b_best_safe.csv', index=False)
log_saved(PRED_DIR / 'test_pred_exp11b_best_safe.csv')

print('[TEST INFERENCE SUMMARY]', flush=True)
log_info(f'selected_gate: {selected_gate_name}')
log_info(f'selected_penalty_variant: {selected_penalty_name}')
log_info(f'n_test_matches: {len(test_pred_match_best):,}')
log_info(f"mean_pred_total: {float((test_pred_match_best['pred_team_a_goals'] + test_pred_match_best['pred_team_b_goals']).mean()):.4f}")
log_info(f"max_pred_goal: {int(test_pred_match_best[['pred_team_a_goals', 'pred_team_b_goals']].max().max())}")

top_scores = test_pred_match_best.assign(
    scoreline=lambda d: d['pred_team_a_goals'].astype(int).astype(str) + '-' + d['pred_team_b_goals'].astype(int).astype(str)
)['scoreline'].value_counts().head(10).reset_index()
top_scores.columns = ['scoreline', 'count']
display(top_scores)


 # 23. Submission Mapping dan Pair Consistency



 Bagian ini mengubah prediksi match-level ke row-level submission. Validasi wajib: shape sama sample,

 Id order sama, goal integer non-negative, tidak ada missing, tidak ada duplicate Id, dan pair consistency.

In [ ]:
def match_predictions_to_submission(test_row_df, pred_match_df):
    row = test_row_df[['Id', 'match_id', 'team']].copy()
    pred = pred_match_df[['match_id', 'team_a', 'team_b', 'pred_team_a_goals', 'pred_team_b_goals']].copy()
    m = row.merge(pred, on='match_id', how='left', validate='many_to_one')
    is_a = m['team'].astype(str) == m['team_a'].astype(str)
    is_b = m['team'].astype(str) == m['team_b'].astype(str)
    assert (is_a | is_b).all(), 'Some test rows cannot be mapped to team_a/team_b.'
    m['team_goals'] = np.where(is_a, m['pred_team_a_goals'], m['pred_team_b_goals']).astype(int)
    m['opp_goals'] = np.where(is_a, m['pred_team_b_goals'], m['pred_team_a_goals']).astype(int)
    sub = sample_submission[['Id']].merge(m[['Id', 'team_goals', 'opp_goals']], on='Id', how='left', validate='one_to_one')
    assert sub['team_goals'].notna().all() and sub['opp_goals'].notna().all()
    assert sub['Id'].tolist() == sample_submission['Id'].tolist()
    return sub[['Id', 'team_goals', 'opp_goals']]


def check_pair_consistency(submission_df, test_df):
    tmp = test_df[['Id', 'match_id', 'team', 'opponent']].merge(submission_df, on='Id', how='left', validate='one_to_one')
    for mid, g in tmp.groupby('match_id'):
        if len(g) != 2:
            return False
        a, b = g.iloc[0], g.iloc[1]
        if int(a['team_goals']) != int(b['opp_goals']):
            return False
        if int(a['opp_goals']) != int(b['team_goals']):
            return False
    return True

submission = match_predictions_to_submission(test_raw, test_pred_match_best)
submission['team_goals'] = submission['team_goals'].clip(lower=0).astype(int)
submission['opp_goals'] = submission['opp_goals'].clip(lower=0).astype(int)
sub_path = SUB_DIR / 'submission_exp11b_best_safe.csv'
submission.to_csv(sub_path, index=False)

log_check('Submission shape matches sample', submission.shape == sample_submission.shape, str(submission.shape))
log_check('Submission Id order matches sample', submission['Id'].tolist() == sample_submission['Id'].tolist())
log_check('Goals are non-negative integers', (submission[['team_goals', 'opp_goals']] >= 0).all().all())
log_check('Submission has no missing', submission.notna().all().all())
log_check('Submission has no duplicate Id', not submission['Id'].duplicated().any())
log_check('Pair consistency', check_pair_consistency(submission, test_raw))
log_saved(sub_path)
display(submission.head())


 # 24. Runtime Summary dan Catatan Akhir



 Bagian ini menyimpan runtime summary dan menampilkan ringkasan akhir. Output akhir yang perlu dicek

 adalah selected penalty config, decision code, validation metric, dan path submission final.

In [ ]:
SECTION_TIMES['total'] = time.time() - NOTEBOOK_START_TIME
runtime_df = pd.DataFrame([{'section': k, 'runtime_sec': float(v)} for k, v in SECTION_TIMES.items()])
runtime_df.to_csv(SUM_DIR / 'notebook_runtime_summary.csv', index=False)
summary_df = pd.DataFrame([
    {'item': 'experiment', 'value': 'EXP11B'},
    {'item': 'selected_gate', 'value': selected_gate_name},
    {'item': 'selected_penalty_variant', 'value': selected_penalty_name},
    {'item': 'decision_code', 'value': decision_code},
    {'item': 'selected_awmae', 'value': float(selected_row['awmae'])},
    {'item': 'submission_path', 'value': str(sub_path)},
])
display(runtime_df)
display(summary_df)
log_saved(SUM_DIR / 'notebook_runtime_summary.csv')
log_result('EXP11B completed. Check validation safety, domain tables, and pair consistency before submitting.')
